# Disaster ML (V4): dataset → training → Raspberry Pi bundle

Flood, landslide, wildfire and extreme-heat early warning. **No forecasting.**

This notebook writes the same Python modules that run on the Raspberry Pi,
then:

1. generates the synthetic V4 dataset (realistic sensor errors, failures and packet loss)
2. trains one model per hazard, calibrates it and chooses WATCH/WARNING thresholds on a validation set
3. scores the models per disaster on locked tests (detection rate, warning time, false alarms per site per day)
4. checks that the live (Pi) features and alerts exactly match training
5. downloads `pi_models.zip` for the Raspberry Pi

Runtime: about 5 minutes on the free Colab CPU.

**Important:** every number here is measured on *synthetic* data. Real-world
accuracy is unknown until you score labelled readings from your own nodes
(the last section).

In [ ]:
import platform, sklearn, numpy, pandas
print("python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
# The Raspberry Pi must install the SAME scikit-learn version to load these
# models; the bundle at the end includes a pinned requirements file.

In [ ]:
import os
os.makedirs("/content/disaster_ml", exist_ok=True)
%cd /content/disaster_ml

## 1. Write the modules (identical to the Raspberry Pi package)

In [ ]:
%%writefile config.py
"""Shared configuration for the Raspberry Pi disaster ML pipeline.

Everything that must stay identical between training and on-device
inference (feature names, window sizes, hazard definitions) lives here.
"""

import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("DISASTER_ML_HOME", Path(__file__).resolve().parent))

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"

DATA_SOURCE = "synthetic_v4"
MODEL_VERSION = "v4"
SAMPLE_INTERVAL_MINUTES = 5
RANDOM_SEED = 42

# Per-reading targets from the Colab pipeline. Still reported, but no longer
# used to pick thresholds: with honest "within 15 min" labels they count an
# alert 45 minutes early as a false positive.
WATCH_RECALL_TARGET = 0.90
WARNING_PRECISION_TARGET = 0.70

# Event-level threshold selection (on the threshold-validation set):
# the most sensitive threshold that stays within a false-alarm budget.
WARNING_MAX_FALSE_ALARMS = 0.2   # per site per day, i.e. about one false WARNING per site every 5 days
WATCH_MAX_FALSE_ALARMS = 1.0
WARNING_MAX_ALERT_TIME = 0.01    # share of non-event time spent in WARNING
WATCH_MAX_ALERT_TIME = 0.05

# Event-level acceptance gates (on locked tests, WARNING level)
MIN_DETECTION_RATE = 0.90
MAX_FALSE_ALARMS_PER_SITE_DAY = 0.5
MAX_ALERT_TIME_OUTSIDE_EVENTS = 0.02
MAX_FAULT_DETECTION_DROP = 0.15
MAX_OOD_DETECTION_DROP = 0.20

# Acceptance gates
MAX_ECE = 0.15
MAX_BRIER_SCORE = 0.25
MIN_PR_AUC_MARGIN_ABOVE_BASELINE = 0.05
MAX_FAULT_RECALL_DROP = 0.15
MAX_OOD_RECALL_DROP = 0.20

# A locked test with fewer disaster episodes than this cannot support a
# PASS/FAIL verdict; the hazard is reported as INSUFFICIENT_EVENTS instead.
MIN_TEST_EPISODES = 20

# Live alerts need this many consecutive readings over the threshold
ALERT_CONFIRM_READINGS = 2

# Hot-day threshold used by hot_window_fraction_1h
HOT_TEMPERATURE_C = 38.0

DATASET_FILES = {
    "train": "disaster_v4_train.csv",
    "calibration": "disaster_v4_calibration.csv",
    "threshold_validation": "disaster_v4_threshold_validation.csv",
    "locked_normal": "disaster_v4_locked_normal.csv",
    "locked_faults": "disaster_v4_locked_faults.csv",
    "locked_ood": "disaster_v4_locked_ood.csv",
}
MANIFEST_FILE = "dataset_v4_manifest.csv"

# A node's history restarts when readings are further apart than this
# (same rule in training features and on the Pi).
MAX_GAP_MINUTES = 3 * SAMPLE_INTERVAL_MINUTES

# Raw sensor fields each reading must carry (missing values are allowed
# and become NaN; HistGradientBoosting handles NaN natively).
RAW_SENSOR_COLUMNS = [
    "water_level_cm",
    "rainfall_mm_h",
    "soil_moisture_pct",
    "temperature_c",
    "humidity_pct",
    "tilt_x_deg",
    "tilt_y_deg",
    "acceleration_g",
    "smoke_raw",
    "gas_raw",
    "flame",
    "node1_flood_score",
    "node1_landslide_score",
    "node1_flood_label",
    "node1_landslide_label",
    "node2_wildfire_score",
    "node2_extreme_heat_score",
    "node2_wildfire_label",
    "node2_extreme_heat_label",
]

# node_id, day_of_week and month were dropped from the Colab feature set:
# each hazard model only ever sees one node (so node_id is constant), and
# day/month only encode the synthetic sites' start dates, not hazard physics.
COMMON_FEATURES = [
    "hour_sin",
    "hour_cos",
]

FLOOD_FEATURES = COMMON_FEATURES + [
    "water_level_cm",
    "water_level_comp_cm",
    "rainfall_mm_h",
    "soil_moisture_pct",
    "temperature_c",
    "humidity_pct",
    "water_level_lag_1",
    "water_level_lag_3",
    "water_level_change_1",
    "water_level_change_3",
    "water_comp_change_1h",
    "water_comp_median_15min",
    "water_comp_std_1h",
    "water_level_anomaly_cm",
    "rainfall_15min_sum",
    "rainfall_1h_sum",
    "rainfall_3h_sum",
    "water_level_15min_mean",
    "water_level_1h_mean",
    "node1_flood_score",
    "node1_flood_label",
]

LANDSLIDE_FEATURES = COMMON_FEATURES + [
    "rainfall_mm_h",
    "soil_moisture_pct",
    "tilt_x_deg",
    "tilt_y_deg",
    "tilt_magnitude_deg",
    "acceleration_g",
    "acceleration_1h_max",
    "soil_moisture_lag_1",
    "soil_moisture_change_1",
    "soil_moisture_change_1h",
    "soil_moisture_anomaly",
    "rainfall_1h_sum",
    "rainfall_3h_sum",
    "tilt_change_1",
    "tilt_change_3",
    "tilt_change_1h",
    "tilt_anomaly_deg",
    "node1_landslide_score",
    "node1_landslide_label",
]

WILDFIRE_FEATURES = COMMON_FEATURES + [
    "temperature_c",
    "humidity_pct",
    "smoke_raw",
    "gas_raw",
    "flame",
    "temperature_lag_1",
    "temperature_change_1",
    "humidity_lag_1",
    "humidity_change_1",
    "smoke_lag_1",
    "smoke_change_1",
    "gas_lag_1",
    "gas_change_1",
    "smoke_15min_mean",
    "gas_15min_mean",
    "smoke_median_15min",
    "gas_median_15min",
    "smoke_ratio_24h",
    "gas_ratio_24h",
    "smoke_change_1h",
    "temperature_anomaly_24h",
    "node2_wildfire_score",
    "node2_wildfire_label",
]

EXTREME_HEAT_FEATURES = COMMON_FEATURES + [
    "temperature_c",
    "humidity_pct",
    "temperature_lag_1",
    "temperature_change_1",
    "humidity_lag_1",
    "humidity_change_1",
    "temperature_1h_mean",
    "temperature_3h_mean",
    "temperature_6h_mean",
    "temperature_6h_max",
    "temperature_24h_mean",
    "temperature_anomaly_24h",
    "temperature_anomaly_72h",
    "temperature_1h_std",
    "temperature_change_24h",
    "humidity_1h_mean",
    "humidity_change_24h",
    "hot_window_fraction_1h",
    "node2_extreme_heat_score",
    "node2_extreme_heat_label",
]

HAZARDS = {
    "flood": {
        "node_id": "node1",
        "target": "flood_within_15m",
        "features": FLOOD_FEATURES,
    },
    "landslide": {
        "node_id": "node1",
        "target": "landslide_within_15m",
        "features": LANDSLIDE_FEATURES,
    },
    "wildfire": {
        "node_id": "node2",
        "target": "wildfire_within_15m",
        "features": WILDFIRE_FEATURES,
    },
    "extreme_heat": {
        "node_id": "node2",
        "target": "extreme_heat_within_60m",
        "features": EXTREME_HEAT_FEATURES,
    },
}

TARGET_COLUMNS = [
    "flood_event",
    "landslide_event",
    "wildfire_event",
    "extreme_heat_event",
    "flood_within_15m",
    "landslide_within_15m",
    "wildfire_within_15m",
    "extreme_heat_within_60m",
]

# Rolling windows in samples (5-minute rows)
WINDOW_15MIN = 4
WINDOW_1H = 12
WINDOW_3H = 36
WINDOW_6H = 72
WINDOW_24H = 288
WINDOW_72H = 864
HISTORY_LENGTH = WINDOW_72H  # longest window the live engine must remember

# Water-level node geometry, used for speed-of-sound compensation. Set these
# to the real installation: transducer height above the channel bed, and the
# air temperature the firmware assumes for the speed of sound.
ULTRASONIC_MOUNT_HEIGHT_CM = 300.0
ULTRASONIC_ASSUMED_AIR_C = 20.0

# Accept the Node.js dashboard's node ids as aliases
NODE_ALIASES = {
    "node1": "node1",
    "node_01": "node1",
    "node01": "node1",
    "node2": "node2",
    "node_02": "node2",
    "node02": "node2",
}


def normalise_node_id(node_id):
    key = str(node_id).strip().lower()
    return NODE_ALIASES.get(key, key)


In [ ]:
%%writefile sensor_errors.py
"""Realistic sensor measurement errors for the synthetic dataset.

The generator simulates the *true* environment. This module turns that into
what each node would actually report, per node and per sensor:

Always on (healthy hardware):
  * calibration: per-unit offset and gain
  * noise and quantisation (sensor resolution, ESP32 12-bit ADC)
  * environmental cross-sensitivity
      - ultrasonic speed of sound vs air temperature (uncompensated)
      - MQ gas sensors vs temperature and humidity
      - capacitive soil probe vs temperature
      - solar heating of the DHT22 enclosure around midday
      - humidity sensor condensation above ~92 % RH
  * tipping-bucket rain gauge: 0.2 mm per tip, undercatch in heavy rain
  * slow drift (MQ baseline, soil probe fouling)
  * ultrasonic echo outliers, blind zone near the transducer
  * ESP32 ADC non-linearity near the top of its range, saturation at 4095
  * DHT22 checksum read failures (NaN)
  * MQ warm-up after power-on
  * IR flame module false triggers in direct sunlight
  * bursty radio packet loss that gets worse in heavy rain; lost rows are
    dropped from the dataset, just like a missed reading on the Pi

Hardware failures (only when ``failures=True``):
  stuck value, disconnected sensor, EMI spike burst, degradation drift
  that keeps growing, clogged rain gauge, low-battery brownout, node
  reboot, communication outage.

The parameters are typical datasheet / field values for the assumed parts
listed in ``HARDWARE``; edit them to match your nodes.
"""

import numpy as np
import pandas as pd

import config as C

STEP_MINUTES = 5
STEPS_PER_DAY = 24 * 60 // STEP_MINUTES

HARDWARE = {
    "water_level_cm": "JSN-SR04T waterproof ultrasonic, mounted above the water",
    "rainfall_mm_h": "tipping-bucket rain gauge, 0.2 mm per tip",
    "soil_moisture_pct": "capacitive soil moisture probe v1.2 on ESP32 ADC",
    "temperature_c": "DHT22",
    "humidity_pct": "DHT22",
    "tilt_x_deg": "MPU6050 (tared at install)",
    "tilt_y_deg": "MPU6050 (tared at install)",
    "acceleration_g": "MPU6050",
    "smoke_raw": "MQ-2 on ESP32 12-bit ADC (0-4095)",
    "gas_raw": "MQ-135 on ESP32 12-bit ADC (0-4095)",
    "flame": "IR flame module, digital output",
}

NODE_SENSORS = {
    "node1": [
        "water_level_cm", "rainfall_mm_h", "soil_moisture_pct", "temperature_c",
        "humidity_pct", "tilt_x_deg", "tilt_y_deg", "acceleration_g",
    ],
    "node2": ["temperature_c", "humidity_pct", "smoke_raw", "gas_raw", "flame"],
}

ALL_SENSORS = [
    "water_level_cm", "rainfall_mm_h", "soil_moisture_pct", "temperature_c", "humidity_pct",
    "tilt_x_deg", "tilt_y_deg", "acceleration_g", "smoke_raw", "gas_raw", "flame",
]

TINYML_COLUMNS = {
    "node1": ["node1_flood_score", "node1_landslide_score", "node1_flood_label", "node1_landslide_label"],
    "node2": ["node2_wildfire_score", "node2_extreme_heat_score", "node2_wildfire_label", "node2_extreme_heat_label"],
}

ULTRASONIC_MOUNT_HEIGHT_CM = C.ULTRASONIC_MOUNT_HEIGHT_CM
ULTRASONIC_BLIND_ZONE_CM = 25.0
ULTRASONIC_MAX_RANGE_CM = 450.0
ULTRASONIC_ASSUMED_AIR_C = C.ULTRASONIC_ASSUMED_AIR_C   # firmware uses a fixed speed of sound
RAIN_MM_PER_TIP = 0.2
ADC_MAX = 4095
ADC_KNEE = 3100                      # ESP32 ADC flattens above ~2.5 V

PHYSICAL_RANGE = {
    "water_level_cm": (0.0, ULTRASONIC_MOUNT_HEIGHT_CM - ULTRASONIC_BLIND_ZONE_CM),
    "rainfall_mm_h": (0.0, 400.0),
    "soil_moisture_pct": (0.0, 100.0),
    "temperature_c": (-40.0, 80.0),   # DHT22 range
    "humidity_pct": (0.0, 100.0),
    "tilt_x_deg": (-90.0, 90.0),
    "tilt_y_deg": (-90.0, 90.0),
    "acceleration_g": (-16.0, 16.0),
    "smoke_raw": (0.0, ADC_MAX),
    "gas_raw": (0.0, ADC_MAX),
}

# Expected failure episodes per node per day, and their duration range in
# 5-minute samples.
FAILURES = {
    "stuck": (0.04, 12, 288),
    "disconnected": (0.02, 12, 576),
    "spike_burst": (0.08, 1, 6),
    "rain_gauge_clog": (0.015, 288, 1440),
    "brownout": (0.04, 12, 72),
    "reboot": (0.05, 1, 4),
    "comm_outage": (0.03, 12, 144),
}
DRIFT_RUNAWAY_PROBABILITY = 0.35     # chance a node develops a growing drift


def speed_of_sound(temperature_c):
    return 331.3 + 0.606 * temperature_c


def esp32_adc(values, rng):
    """ADC noise, top-end compression, saturation and integer counts."""
    values = values + rng.normal(0, 12, values.shape)
    values = np.where(values > ADC_KNEE, ADC_KNEE + (values - ADC_KNEE) * 0.45, values)
    return np.round(np.clip(values, 0, ADC_MAX))


def quantise(values, step):
    return np.round(values / step) * step


def gilbert_elliott_loss(rain_mm_h, rng, base_bad=0.006, rain_bad=0.0006, recover=0.45):
    """Bursty packet loss; heavy rain attenuates the radio link."""
    n = len(rain_mm_h)
    lost = np.zeros(n, dtype=bool)
    bad = False
    draws = rng.random(n)
    loss_draws = rng.random(n)
    for i in range(n):
        if bad:
            bad = draws[i] >= recover
        else:
            bad = draws[i] < base_bad + rain_bad * rain_mm_h[i]
        lost[i] = bad and loss_draws[i] < 0.9
    return lost


def warmup_factor(n, boot_steps):
    """MQ heater warm-up: readings start ~2.5x high and settle in ~30 min."""
    factor = np.ones(n)
    for boot in boot_steps:
        k = np.arange(n - boot)
        factor[boot:] = np.maximum(factor[boot:], 1.0 + 1.5 * np.exp(-k / 2.5))
    return factor


def schedule_failures(n, rng, node_sensors, scale=1.0):
    """Return list of (type, sensor, start, end) failure episodes."""
    days = n / STEPS_PER_DAY
    episodes = []
    for failure, (rate, low, high) in FAILURES.items():
        if failure == "rain_gauge_clog" and "rainfall_mm_h" not in node_sensors:
            continue
        for _ in range(rng.poisson(rate * days * scale)):
            start = int(rng.integers(0, n))
            end = min(n, start + int(rng.integers(low, high + 1)))
            sensor = None
            if failure in ("stuck", "disconnected", "spike_burst"):
                sensor = str(rng.choice([s for s in node_sensors if s != "flame"]))
            episodes.append((failure, sensor, start, end))
    if rng.random() < min(1.0, DRIFT_RUNAWAY_PROBABILITY * scale):
        sensor = str(rng.choice([s for s in node_sensors if s in
                                 ("water_level_cm", "soil_moisture_pct", "temperature_c", "smoke_raw", "gas_raw")]))
        episodes.append(("drift_runaway", sensor, int(rng.integers(0, n)), n))
    return episodes


def measure_node(true_df, node_id, rng, failures, failure_scale=1.0):
    """Measured readings for one node at one site.

    ``true_df`` holds the true environment for consecutive 5-minute steps.
    Returns (measured DataFrame with only this node's sensors filled,
    per-row fault labels, boolean mask of rows that were delivered).
    """
    n = len(true_df)
    sensors = NODE_SENSORS[node_id]
    timestamps = pd.to_datetime(true_df["timestamp"])
    hour = (timestamps.dt.hour + timestamps.dt.minute / 60).to_numpy()
    day_index = np.arange(n) // STEPS_PER_DAY
    days_elapsed = np.arange(n) / STEPS_PER_DAY

    t_air = true_df["temperature_c"].to_numpy(float)
    rh_air = true_df["humidity_pct"].to_numpy(float)
    rain = true_df["rainfall_mm_h"].to_numpy(float)

    measured = {}
    fault_labels = [[] for _ in range(n)]

    # Daily sunshine (0 = overcast, 1 = clear) drives enclosure heating
    sunshine = rng.uniform(0, 1, day_index.max() + 1)[day_index]
    solar_shape = np.clip(np.sin(np.pi * (hour - 6) / 12), 0, None)
    solar_bias = rng.uniform(1.0, 4.0) * sunshine * solar_shape

    # ---- DHT22 temperature / humidity ----------------------------------
    temperature = t_air + rng.normal(0, 0.3) + solar_bias + rng.normal(0, 0.2, n)
    humidity = rh_air + rng.normal(0, 2.0) - 3.0 * solar_bias + rng.normal(0, 1.0, n)
    humidity = humidity + np.where(rh_air > 92, rng.uniform(2, 6, n), 0)
    dht_fail = rng.random(n) < 0.01
    measured["temperature_c"] = np.where(dht_fail, np.nan, quantise(temperature, 0.1))
    measured["humidity_pct"] = np.where(dht_fail, np.nan, quantise(np.clip(humidity, 0, 100), 0.1))

    if node_id == "node1":
        # ---- Ultrasonic water level --------------------------------------
        level = true_df["water_level_cm"].to_numpy(float)
        distance = ULTRASONIC_MOUNT_HEIGHT_CM - level
        distance = distance * speed_of_sound(ULTRASONIC_ASSUMED_AIR_C) / speed_of_sound(t_air)
        distance = distance + rng.normal(0, 0.4 + 0.03 * rain, n)  # surface ripples in rain
        echo_outlier = rng.random(n) < 0.004
        distance = np.where(echo_outlier, rng.uniform(ULTRASONIC_BLIND_ZONE_CM, ULTRASONIC_MAX_RANGE_CM, n), distance)
        in_blind_zone = distance < ULTRASONIC_BLIND_ZONE_CM
        distance = np.where(in_blind_zone, np.nan, np.minimum(distance, ULTRASONIC_MAX_RANGE_CM))
        measured["water_level_cm"] = quantise(np.clip(ULTRASONIC_MOUNT_HEIGHT_CM - distance, 0, None), 0.1)

        # ---- Tipping bucket ----------------------------------------------
        catch = np.clip(0.95 - 0.0015 * rain, 0.7, 1.0)
        collected = np.cumsum(rain * STEP_MINUTES / 60 * catch)
        tips = np.diff(np.floor(collected / RAIN_MM_PER_TIP), prepend=0)
        measured["rainfall_mm_h"] = tips * RAIN_MM_PER_TIP * 60 / STEP_MINUTES

        # ---- Capacitive soil moisture ------------------------------------
        soil = true_df["soil_moisture_pct"].to_numpy(float)
        soil = rng.uniform(0.85, 1.15) * soil + rng.normal(0, 4.0)
        soil = soil - 0.25 * (t_air - 25) + rng.uniform(0, 0.1) * days_elapsed + rng.normal(0, 0.8, n)
        measured["soil_moisture_pct"] = quantise(np.clip(soil, 0, 100), 0.1)

        # ---- MPU6050 -----------------------------------------------------
        accel_true = true_df["acceleration_g"].to_numpy(float)
        vibration = np.clip(accel_true - 1.0, 0, None)
        for axis in ("tilt_x_deg", "tilt_y_deg"):
            tilt = true_df[axis].to_numpy(float) + rng.normal(0, 0.15)
            tilt = tilt + 0.01 * (t_air - 25) + rng.normal(0, 0.05 + 2.0 * vibration, n)
            measured[axis] = quantise(tilt, 0.01)
        measured["acceleration_g"] = quantise(
            accel_true + rng.normal(0, 0.015) + rng.normal(0, 0.004, n), 0.001
        )

    else:
        # ---- MQ-2 / MQ-135 on the ESP32 ADC -----------------------------
        environment = 1 + 0.004 * (rh_air - 65) + 0.006 * (t_air - 25)
        for column in ("smoke_raw", "gas_raw"):
            true_value = true_df[column].to_numpy(float)
            gain = rng.lognormal(0, 0.25)
            drift = 1 + rng.uniform(0, 0.015) * days_elapsed  # baseline creep, up to 1.5 %/day
            value = (true_value * gain + rng.normal(0, 30)) * environment * drift
            measured[column] = value * (1 + rng.normal(0, 0.02, n)) + rng.normal(0, 6, n)

        # ---- IR flame module --------------------------------------------
        flame = true_df["flame"].to_numpy(float)
        detected = (flame > 0) & (rng.random(n) > 0.3)
        sun_trigger = (rng.random(n) < 0.003 * sunshine) & (hour >= 9) & (hour <= 16)
        measured["flame"] = (detected | sun_trigger).astype(float)

    # ---- Hardware failures --------------------------------------------------
    delivered = ~gilbert_elliott_loss(rain, rng)
    boot_steps = [0]

    if failures:
        for failure, sensor, start, end in schedule_failures(n, rng, sensors, failure_scale):
            window = slice(start, end)
            if failure == "stuck":
                held = measured[sensor][start]
                measured[sensor][window] = held
            elif failure == "disconnected":
                if sensor in ("temperature_c", "humidity_pct"):
                    measured[sensor][window] = np.nan
                elif sensor == "water_level_cm":
                    measured[sensor][window] = 0.0  # no echo -> max range -> level 0
                else:
                    measured[sensor][window] = 0.0  # floating / grounded ADC pin
            elif failure == "spike_burst":
                values = measured[sensor]
                low, high = np.nanpercentile(values, 1), np.nanpercentile(values, 99)
                values[window] = rng.uniform(low, high * 1.8 + 1, end - start)
            elif failure == "drift_runaway":
                ramp = (np.arange(n - start) / STEPS_PER_DAY) * {
                    "water_level_cm": 1.5,     # condensation / spider web on transducer
                    "soil_moisture_pct": 0.8,  # probe coating degradation
                    "temperature_c": 0.25,     # dirty radiation shield
                    "smoke_raw": 40.0,         # MQ sensor poisoning
                    "gas_raw": 50.0,
                }[sensor]
                measured[sensor][start:] = measured[sensor][start:] + ramp
            elif failure == "rain_gauge_clog":
                measured["rainfall_mm_h"][window] = 0.0
            elif failure == "brownout":
                for analog in ("soil_moisture_pct", "smoke_raw", "gas_raw"):
                    if analog in measured:
                        scale = rng.uniform(0.8, 0.92)
                        noisy = measured[analog][window] * scale
                        measured[analog][window] = noisy + rng.normal(0, 5, end - start)
            elif failure == "reboot":
                delivered[window] = False
                boot_steps.append(end)
            elif failure == "comm_outage":
                delivered[window] = False

            label = failure if sensor is None else f"{failure}:{sensor}"
            for index in range(start, end):
                fault_labels[index].append(label)

    if node_id == "node2":
        warm = warmup_factor(n, boot_steps)
        for column in ("smoke_raw", "gas_raw"):
            measured[column] = esp32_adc(measured[column] * warm, rng)

    # Nothing can report outside what the hardware is able to output
    for sensor, (low, high) in PHYSICAL_RANGE.items():
        if sensor in measured:
            measured[sensor] = np.clip(measured[sensor], low, high)

    frame = pd.DataFrame({column: np.full(n, np.nan) for column in ALL_SENSORS})
    for sensor in sensors:
        frame[sensor] = measured[sensor]

    labels = np.array(["+".join(f) if f else "normal" for f in fault_labels], dtype=object)
    return frame, labels, delivered


def ideal_node(true_df, node_id):
    """Perfect sensors: measured == true for this node's sensors."""
    n = len(true_df)
    frame = pd.DataFrame({column: np.full(n, np.nan) for column in ALL_SENSORS})
    for sensor in NODE_SENSORS[node_id]:
        frame[sensor] = true_df[sensor].to_numpy(float)
    return frame, np.array(["normal"] * n, dtype=object), np.ones(n, dtype=bool)


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def tinyml_scores(measured, node_id, rng):
    """On-node TinyML scores computed from what the node measured.

    Firmware holds the last good reading when a sensor read fails, so NaNs
    are forward-filled; before the first good reading the score is NaN.
    """
    m = measured.ffill()
    n = len(m)
    out = {}
    if node_id == "node1":
        tilt = np.sqrt(m["tilt_x_deg"] ** 2 + m["tilt_y_deg"] ** 2)
        flood = _sigmoid(-6.0 + 0.055 * m["water_level_cm"] + 0.045 * m["rainfall_mm_h"]
                         + 0.015 * m["soil_moisture_pct"] + rng.normal(0, 1.1, n))
        landslide = _sigmoid(-7.0 + 0.035 * m["soil_moisture_pct"] + 0.25 * tilt
                             + 1.20 * np.clip(m["acceleration_g"] - 1.0, 0, None)
                             + 0.020 * m["rainfall_mm_h"] + rng.normal(0, 1.1, n))
        scores = {"node1_flood": flood, "node1_landslide": landslide}
    else:
        wildfire = _sigmoid(-7.0 + 0.045 * m["temperature_c"] - 0.014 * m["humidity_pct"]
                            + 0.0030 * m["smoke_raw"] + 0.0010 * m["gas_raw"]
                            + 0.75 * m["flame"] + rng.normal(0, 1.2, n))
        heat = _sigmoid(-9.0 + 0.16 * m["temperature_c"] - 0.012 * m["humidity_pct"]
                        + rng.normal(0, 1.15, n))
        scores = {"node2_wildfire": wildfire, "node2_extreme_heat": heat}

    for column in sum(TINYML_COLUMNS.values(), []):
        out[column] = np.full(n, np.nan)
    for name, score in scores.items():
        score = np.asarray(score, dtype=float)
        out[f"{name}_score"] = score
        out[f"{name}_label"] = np.where(np.isnan(score), np.nan, (score >= 0.60).astype(float))
    return pd.DataFrame(out)


In [ ]:
%%writefile features.py
"""Causal feature engineering, in two forms that must agree exactly.

* ``prepare_dataframe``      - vectorised pandas version (training, CSV evaluation)
* ``StreamingFeatureEngine`` - dependency-light version for live readings on
  the Pi; keeps the last ``HISTORY_LENGTH`` readings per (site, node)

Both are driven by the same specs (``DERIVED``, ``LAGS``, ``ROLLING``), and
``test_feature_parity.py`` checks they produce the same numbers.

Windows count *readings*. Both versions restart a node's history when the
gap between readings exceeds ``config.MAX_GAP_MINUTES``.
"""

import math
import statistics
from collections import deque
from datetime import datetime, timezone

import numpy as np

import config as C

GROUP_COLUMNS = ["site_id", "node_id", "_segment"]

W15, W1H, W3H, W6H, W24H, W72H = (C.WINDOW_15MIN, C.WINDOW_1H, C.WINDOW_3H, C.WINDOW_6H,
                                  C.WINDOW_24H, C.WINDOW_72H)

# (feature name, source column, steps back); "<name>_change_<k>" features are
# derived as current minus lag, see CHANGES.
LAGS = [
    ("water_level_lag_1", "water_level_cm", 1),
    ("water_level_lag_3", "water_level_cm", 3),
    ("soil_moisture_lag_1", "soil_moisture_pct", 1),
    ("temperature_lag_1", "temperature_c", 1),
    ("humidity_lag_1", "humidity_pct", 1),
    ("smoke_lag_1", "smoke_raw", 1),
    ("gas_lag_1", "gas_raw", 1),
    ("tilt_lag_1", "tilt_magnitude_deg", 1),
    ("tilt_lag_3", "tilt_magnitude_deg", 3),
    ("water_comp_lag_12", "water_level_comp_cm", W1H),
    ("soil_moisture_lag_12", "soil_moisture_pct", W1H),
    ("tilt_lag_12", "tilt_magnitude_deg", W1H),
    ("smoke_median_lag_12", "smoke_median_15min", W1H),
    # Same time yesterday: removes the day/night cycle for heat waves
    ("temperature_1h_mean_lag_24h", "temperature_1h_mean", W24H),
    ("humidity_1h_mean_lag_24h", "humidity_1h_mean", W24H),
]

# (feature name, column, lag feature)
CHANGES = [
    ("water_level_change_1", "water_level_cm", "water_level_lag_1"),
    ("water_level_change_3", "water_level_cm", "water_level_lag_3"),
    ("soil_moisture_change_1", "soil_moisture_pct", "soil_moisture_lag_1"),
    ("temperature_change_1", "temperature_c", "temperature_lag_1"),
    ("humidity_change_1", "humidity_pct", "humidity_lag_1"),
    ("smoke_change_1", "smoke_raw", "smoke_lag_1"),
    ("gas_change_1", "gas_raw", "gas_lag_1"),
    ("tilt_change_1", "tilt_magnitude_deg", "tilt_lag_1"),
    ("tilt_change_3", "tilt_magnitude_deg", "tilt_lag_3"),
    ("water_comp_change_1h", "water_level_comp_cm", "water_comp_lag_12"),
    ("soil_moisture_change_1h", "soil_moisture_pct", "soil_moisture_lag_12"),
    ("tilt_change_1h", "tilt_magnitude_deg", "tilt_lag_12"),
    ("smoke_change_1h", "smoke_median_15min", "smoke_median_lag_12"),
    ("temperature_change_24h", "temperature_1h_mean", "temperature_1h_mean_lag_24h"),
    ("humidity_change_24h", "humidity_1h_mean", "humidity_1h_mean_lag_24h"),
]

# Rolling windows, computed in this order (later specs may use earlier ones).
# (feature name, column, window, statistic)
ROLLING_STAGE_1 = [
    ("rainfall_15min_sum", "rainfall_mm_h", W15, "sum"),
    ("rainfall_1h_sum", "rainfall_mm_h", W1H, "sum"),
    ("rainfall_3h_sum", "rainfall_mm_h", W3H, "sum"),
    ("water_level_15min_mean", "water_level_cm", W15, "mean"),
    ("water_level_1h_mean", "water_level_cm", W1H, "mean"),
    ("water_comp_median_15min", "water_level_comp_cm", W15, "median"),
    ("water_comp_std_1h", "water_level_comp_cm", W1H, "std"),
    ("soil_moisture_24h_mean", "soil_moisture_pct", W24H, "mean"),
    ("tilt_24h_median", "tilt_magnitude_deg", W24H, "median"),
    ("acceleration_1h_max", "acceleration_g", W1H, "max"),
    ("smoke_15min_mean", "smoke_raw", W15, "mean"),
    ("gas_15min_mean", "gas_raw", W15, "mean"),
    ("smoke_median_15min", "smoke_raw", W15, "median"),
    ("gas_median_15min", "gas_raw", W15, "median"),
    ("smoke_24h_median", "smoke_raw", W24H, "median"),
    ("gas_24h_median", "gas_raw", W24H, "median"),
    ("temperature_1h_mean", "temperature_c", W1H, "mean"),
    ("temperature_3h_mean", "temperature_c", W3H, "mean"),
    ("temperature_6h_mean", "temperature_c", W6H, "mean"),
    ("temperature_6h_max", "temperature_c", W6H, "max"),
    ("temperature_24h_mean", "temperature_c", W24H, "mean"),
    ("temperature_72h_mean", "temperature_c", W72H, "mean"),
    ("temperature_1h_std", "temperature_c", W1H, "std"),
    ("humidity_1h_mean", "humidity_pct", W1H, "mean"),
]

# Ratios against a slow baseline cancel per-unit gain and baseline drift of
# MQ sensors; +10 keeps a disconnected (0) sensor from dividing by zero.
RATIO_OFFSET = 10.0


def _speed_of_sound(temperature_c):
    return 331.3 + 0.606 * temperature_c


def compensate_water_level(level_cm, temperature_c):
    """Undo the ultrasonic speed-of-sound error using the node's own DHT22."""
    height = C.ULTRASONIC_MOUNT_HEIGHT_CM
    factor = _speed_of_sound(temperature_c) / _speed_of_sound(C.ULTRASONIC_ASSUMED_AIR_C)
    return height - (height - level_cm) * factor


def _finish(features):
    """Features computed from other features (same code for both paths)."""
    features["water_level_anomaly_cm"] = features["water_comp_median_15min"] - features["water_comp_24h_mean"]
    features["soil_moisture_anomaly"] = features["soil_moisture_pct"] - features["soil_moisture_24h_mean"]
    features["tilt_anomaly_deg"] = features["tilt_magnitude_deg"] - features["tilt_24h_median"]
    features["smoke_ratio_24h"] = (features["smoke_median_15min"] + RATIO_OFFSET) / (
        features["smoke_24h_median"] + RATIO_OFFSET)
    features["gas_ratio_24h"] = (features["gas_median_15min"] + RATIO_OFFSET) / (
        features["gas_24h_median"] + RATIO_OFFSET)
    features["temperature_anomaly_24h"] = features["temperature_1h_mean"] - features["temperature_24h_mean"]
    features["temperature_anomaly_72h"] = features["temperature_3h_mean"] - features["temperature_72h_mean"]
    return features


ROLLING_STAGE_2 = [
    ("water_comp_24h_mean", "water_comp_median_15min", W24H, "mean"),
]


# ----------------------------------------------------------------------
# Batch (pandas) version
# ----------------------------------------------------------------------

def prepare_dataframe(dataframe):
    import pandas as pd

    dataframe = dataframe.copy()

    dataframe["timestamp"] = pd.to_datetime(dataframe["timestamp"], errors="coerce", utc=True)
    dataframe = dataframe.dropna(subset=["timestamp", "site_id", "node_id"]).copy()
    dataframe["site_id"] = dataframe["site_id"].astype(str)
    dataframe["node_id"] = dataframe["node_id"].map(C.normalise_node_id)

    for column in C.RAW_SENSOR_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = np.nan
        dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce")

    dataframe = dataframe.sort_values(["site_id", "node_id", "timestamp"]).reset_index(drop=True)

    for column in C.TARGET_COLUMNS:
        if column in dataframe.columns:
            dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce").fillna(0).astype(int).clip(0, 1)

    gap_minutes = (
        dataframe.groupby(["site_id", "node_id"], sort=False)["timestamp"].diff().dt.total_seconds() / 60
    )
    dataframe["_segment"] = (gap_minutes.isna() | (gap_minutes > C.MAX_GAP_MINUTES)).cumsum()

    hour = dataframe["timestamp"].dt.hour
    dataframe["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    dataframe["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    dataframe["tilt_magnitude_deg"] = np.sqrt(dataframe["tilt_x_deg"] ** 2 + dataframe["tilt_y_deg"] ** 2)
    compensated = compensate_water_level(dataframe["water_level_cm"], dataframe["temperature_c"])
    dataframe["water_level_comp_cm"] = compensated.fillna(dataframe["water_level_cm"])

    def rolling(column, window, how):
        grouped = dataframe.groupby(GROUP_COLUMNS, sort=False)[column]
        result = getattr(grouped.rolling(window=window, min_periods=1), how)()
        return result.reset_index(level=GROUP_COLUMNS, drop=True)

    for name, column, window, how in ROLLING_STAGE_1 + ROLLING_STAGE_2:
        dataframe[name] = rolling(column, window, how)

    grouped = dataframe.groupby(GROUP_COLUMNS, sort=False)
    for name, column, steps in LAGS:
        dataframe[name] = grouped[column].shift(steps)
    for name, column, lag in CHANGES:
        dataframe[name] = dataframe[column] - dataframe[lag]

    # Share of hot readings in the last hour; missing readings count in the
    # denominator but never as hot, an hour with no valid reading is NaN.
    dataframe["_hot"] = (dataframe["temperature_c"] >= C.HOT_TEMPERATURE_C).astype(float)
    dataframe["_valid_temp"] = dataframe["temperature_c"].notna().astype(float)
    hot_count = rolling("_hot", W1H, "sum")
    valid_count = rolling("_valid_temp", W1H, "sum")
    rows_in_window = (dataframe.groupby(GROUP_COLUMNS, sort=False).cumcount() + 1).clip(upper=W1H)
    dataframe["hot_window_fraction_1h"] = np.where(valid_count > 0, hot_count / rows_in_window, np.nan)

    _finish(dataframe)
    return dataframe.drop(columns=["_hot", "_valid_temp", "_segment"])


def feature_matrix(dataframe, feature_columns):
    """Float32 matrix in the exact column order the model was trained on."""
    return dataframe[feature_columns].to_numpy(dtype=np.float32)


# ----------------------------------------------------------------------
# Streaming version for live inference on the Pi
# ----------------------------------------------------------------------

def _to_float(value):
    if value is None:
        return math.nan
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan


def _parse_timestamp(value):
    if value is None:
        return datetime.now(timezone.utc)
    if isinstance(value, datetime):
        timestamp = value
    else:
        timestamp = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    if timestamp.tzinfo is None:
        timestamp = timestamp.replace(tzinfo=timezone.utc)
    return timestamp.astimezone(timezone.utc)


def _valid(values):
    return [v for v in values if not math.isnan(v)]


def _stat(values, how):
    valid = _valid(values)
    if how == "std":
        return statistics.stdev(valid) if len(valid) >= 2 else math.nan
    if not valid:
        return math.nan
    if how == "sum":
        return math.fsum(valid)
    if how == "mean":
        return math.fsum(valid) / len(valid)
    if how == "max":
        return max(valid)
    if how == "median":
        return statistics.median(valid)
    raise ValueError(how)


class StreamingFeatureEngine:
    """Keeps a rolling history per (site, node) and emits features."""

    def __init__(self, max_gap_minutes=C.MAX_GAP_MINUTES):
        self.max_gap_minutes = max_gap_minutes
        self.history = {}
        self.last_timestamp = {}

    def update(self, reading):
        """Add one reading (dict) and return (node_id, features dict)."""
        node_id = C.normalise_node_id(reading.get("node_id", ""))
        site_id = str(reading.get("site_id", "default_site"))
        timestamp = _parse_timestamp(reading.get("timestamp"))
        key = (site_id, node_id)

        previous = self.last_timestamp.get(key)
        if previous is not None and self.max_gap_minutes is not None:
            gap = (timestamp - previous).total_seconds() / 60.0
            if gap < 0 or gap > self.max_gap_minutes:
                self.history.pop(key, None)
        self.last_timestamp[key] = timestamp

        row = {column: _to_float(reading.get(column)) for column in C.RAW_SENSOR_COLUMNS}
        row["tilt_magnitude_deg"] = math.sqrt(row["tilt_x_deg"] ** 2 + row["tilt_y_deg"] ** 2)
        compensated = compensate_water_level(row["water_level_cm"], row["temperature_c"])
        row["water_level_comp_cm"] = row["water_level_cm"] if math.isnan(compensated) else compensated
        hour = timestamp.hour
        row["hour_sin"] = math.sin(2 * math.pi * hour / 24)
        row["hour_cos"] = math.cos(2 * math.pi * hour / 24)

        history = self.history.setdefault(key, deque(maxlen=C.HISTORY_LENGTH))
        history.append(row)

        # Rolling features are stored on the row so later stages (and lags of
        # rolling features) can read them from history.
        rows = list(history)
        for stage in (ROLLING_STAGE_1, ROLLING_STAGE_2):
            for name, column, window, how in stage:
                row[name] = _stat([r[column] for r in rows[-window:]], how)

        features = dict(row)
        for name, column, steps in LAGS:
            features[name] = rows[-1 - steps][column] if len(rows) > steps else math.nan
        for name, column, lag in CHANGES:
            features[name] = features[column] - features[lag]

        temps = [r["temperature_c"] for r in rows[-W1H:]]
        if all(math.isnan(t) for t in temps):
            features["hot_window_fraction_1h"] = math.nan
        else:
            features["hot_window_fraction_1h"] = sum(1 for t in temps if t >= C.HOT_TEMPERATURE_C) / len(temps)

        return node_id, _finish(features)


In [ ]:
%%writefile metrics.py
"""Classification metrics and WATCH/WARNING threshold selection."""

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import config as C


def expected_calibration_error(y_true, probabilities, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probabilities, bins[1:-1], right=True)

    ece_value = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if mask.sum() == 0:
            continue
        ece_value += mask.mean() * abs(y_true[mask].mean() - probabilities[mask].mean())
    return float(ece_value)


def calculate_classification_metrics(y_true, probabilities, threshold):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    has_both_classes = len(np.unique(y_true)) == 2

    return {
        "threshold": float(threshold),
        "samples": int(len(y_true)),
        "positive_samples": int(y_true.sum()),
        "negative_samples": int((y_true == 0).sum()),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "pr_auc": float(average_precision_score(y_true, probabilities)) if y_true.sum() else np.nan,
        "roc_auc": float(roc_auc_score(y_true, probabilities)) if has_both_classes else np.nan,
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "ece": expected_calibration_error(y_true, probabilities),
        "positive_rate_baseline": float(y_true.mean()),
        "always_negative_accuracy": float((y_true == 0).mean()),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else np.nan,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else np.nan,
    }


PRINT_KEYS = [
    "threshold", "accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc",
    "brier_score", "ece", "true_negative", "false_positive", "false_negative",
    "true_positive", "false_positive_rate", "false_negative_rate",
    "always_negative_accuracy",
]


def print_classification_metrics(title, metrics):
    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)
    for key in PRINT_KEYS:
        value = metrics[key]
        print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")


def select_ordered_thresholds(y_true, probabilities):
    """Same rule as the Colab pipeline.

    WATCH   = highest threshold whose recall >= WATCH_RECALL_TARGET.
    WARNING = threshold >= WATCH with precision >= WARNING_PRECISION_TARGET
              and the best recall.
    When WATCH already meets the precision target this returns
    WARNING == WATCH (the Colab run hit this for every hazard).
    The candidate grid runs from 0.0001 to 0.99.
    """
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=float)
    # Fine steps below 0.01 too: for rare events the right threshold on a
    # calibrated probability is often tiny (the Colab grid stopped at 0.01).
    grid = np.unique(np.round(np.concatenate([np.geomspace(1e-4, 0.01, 41), np.linspace(0.01, 0.99, 99)]), 6))

    def score(threshold):
        prediction = (probabilities >= threshold).astype(int)
        return (
            float(recall_score(y_true, prediction, zero_division=0)),
            float(precision_score(y_true, prediction, zero_division=0)),
        )

    scored = [(float(t), *score(t)) for t in grid]

    watch = [s for s in scored if s[1] >= C.WATCH_RECALL_TARGET]
    if not watch:
        return {"status": "FAIL", "reason": "No threshold satisfies WATCH recall target."}
    watch_threshold, watch_recall, watch_precision = max(watch, key=lambda s: (s[0], s[2]))

    warning = [s for s in scored if s[0] >= watch_threshold and s[2] >= C.WARNING_PRECISION_TARGET]
    if not warning:
        return {
            "status": "FAIL",
            "reason": "No WARNING threshold >= WATCH threshold satisfies precision target.",
            "watch_threshold": watch_threshold,
        }
    warning_threshold, warning_recall, warning_precision = max(warning, key=lambda s: (s[1], -s[0]))

    return {
        "status": "PASS",
        "watch_threshold": watch_threshold,
        "warning_threshold": warning_threshold,
        "watch_precision": watch_precision,
        "watch_recall": watch_recall,
        "warning_precision": warning_precision,
        "warning_recall": warning_recall,
    }


# ----------------------------------------------------------------------
# Event-level scoring (what an operator experiences)
# ----------------------------------------------------------------------

PRECURSOR_PHASES = ("pre_event", "watch", "warning", "event")


def runs(mask):
    """(start, end) index pairs of contiguous True runs."""
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return []
    edges = np.diff(np.concatenate([[0], mask.astype(int), [0]]))
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))


def confirm_alerts(raw_alert, confirm):
    """Alert only after ``confirm`` consecutive readings over the threshold."""
    raw_alert = np.asarray(raw_alert, dtype=bool)
    if confirm <= 1:
        return raw_alert
    streak = np.zeros(len(raw_alert), dtype=int)
    for i, value in enumerate(raw_alert):
        streak[i] = (streak[i - 1] + 1) if (value and i) else int(value)
    return streak >= confirm


def site_groups(subset, hazard):
    """Precompute per-site arrays used by score_events."""
    groups = []
    for _, index in subset.groupby("site_id").indices.items():
        site = subset.iloc[index]
        subtype = site["event_subtype"].to_numpy(dtype=str)
        phase = site["event_phase"].to_numpy(dtype=str)
        in_hazard = np.char.find(subtype, hazard) >= 0
        episode_mask = in_hazard & np.isin(phase, PRECURSOR_PHASES)
        times = site["timestamp"].to_numpy()
        groups.append({
            "index": index,
            "times": times,
            "phase": phase,
            "excused": in_hazard,  # episode and its recovery
            "episodes": runs(episode_mask),
            "days": (times[-1] - times[0]) / np.timedelta64(1, "D"),
        })
    return groups


def score_events(groups, probabilities, threshold, confirm=1):
    """Detection rate, early warning, lead time and false alarms per site-day.

    An episode is one disaster's pre_event..event rows. It is detected if any
    alert falls inside it. Every separate stretch of alerting outside all
    episodes and their recoveries is one false alarm, and
    ``alert_time_outside_events`` is the share of non-event time spent alerting.
    """
    episodes = detected = early = false_alarms = false_alert_rows = normal_rows = 0
    leads = []
    days = 0.0
    for group in groups:
        alert = confirm_alerts(probabilities[group["index"]] >= threshold, confirm)
        days += group["days"]
        for start, end in group["episodes"]:
            episodes += 1
            hits = np.flatnonzero(alert[start:end])
            if not len(hits):
                continue
            detected += 1
            event_rows = np.flatnonzero(group["phase"][start:end] == "event")
            if len(event_rows):
                lead = (group["times"][start + event_rows[0]] - group["times"][start + hits[0]]) / np.timedelta64(1, "m")
                leads.append(float(lead))
                early += lead >= 0
        # Count alert stretches outside episodes/recoveries. (Counting whole alert
        # runs that merely touch an episode would let an always-on alarm score
        # zero false alarms.)
        outside = alert & ~group["excused"]
        false_alarms += len(runs(outside))
        false_alert_rows += int(outside.sum())
        normal_rows += int((~group["excused"]).sum())
    return {
        "episodes": episodes,
        "detection_rate": detected / episodes if episodes else np.nan,
        "warned_before_event": early / episodes if episodes else np.nan,
        "median_lead_min": float(np.median(leads)) if leads else np.nan,
        "false_alarms_per_site_day": false_alarms / days if days else np.nan,
        "alert_time_outside_events": false_alert_rows / normal_rows if normal_rows else np.nan,
    }


def select_event_thresholds(groups, probabilities, confirm=1):
    """WATCH / WARNING = most sensitive thresholds within a false-alarm budget.

    Chosen on the threshold-validation set:
      WARNING: lowest threshold with <= WARNING_MAX_FALSE_ALARMS per site-day
               and <= WARNING_MAX_ALERT_TIME of non-event time in alert
      WATCH:   same with the WATCH budgets, never above WARNING
    Both limits are needed: counting alarms alone lets an always-on alarm
    pass (one endless alarm counts once).
    """
    grid = np.unique(np.round(np.concatenate([np.geomspace(1e-4, 0.01, 41), np.linspace(0.01, 0.99, 99)]), 6))
    scored = [(float(t), score_events(groups, probabilities, t, confirm)) for t in grid]

    def lowest_within(max_alarms, max_alert_time):
        for threshold, score in scored:  # ascending: first is most sensitive
            if (score["false_alarms_per_site_day"] <= max_alarms
                    and score["alert_time_outside_events"] <= max_alert_time):
                return threshold, score
        return None, None

    warning_threshold, warning_score = lowest_within(C.WARNING_MAX_FALSE_ALARMS, C.WARNING_MAX_ALERT_TIME)
    if warning_threshold is None:
        return {"status": "FAIL", "reason": "No threshold keeps WARNING false alarms within budget."}
    watch_threshold, watch_score = lowest_within(C.WATCH_MAX_FALSE_ALARMS, C.WATCH_MAX_ALERT_TIME)
    if watch_threshold is None or watch_threshold > warning_threshold:
        watch_threshold, watch_score = warning_threshold, warning_score

    return {
        "status": "PASS",
        "watch_threshold": watch_threshold,
        "warning_threshold": warning_threshold,
        "watch_validation": watch_score,
        "warning_validation": warning_score,
    }


In [ ]:
%%writefile generate_dataset.py
"""Synthetic V4 dataset generator.

The environment simulation (weather, events, hard negatives) is the Colab
V3 logic unchanged. What changed in V4 is the sensor side: each node now
reports its own readings through ``sensor_errors.py`` (noise, calibration,
drift, cross-sensitivity, quantisation, failures, packet loss) instead of
both nodes sharing one perfect set of values.

Usage:
    python3 generate_dataset.py            # full size (~5 min on a Pi 4)
    python3 generate_dataset.py --quick    # small smoke-test dataset
"""

import argparse
import math
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

import config as C
import sensor_errors as SE

RANDOM_SEED = 20260925
rng = np.random.default_rng(RANDOM_SEED)          # true environment
error_rng = np.random.default_rng(RANDOM_SEED + 1)  # sensor errors, separate so
                                                    # ideal vs realistic share the same truth

ROWS_PER_DAY = 288


def clip(value, low, high):
    return float(np.clip(value, low, high))


def sigmoid(value):
    return float(1.0 / (1.0 + np.exp(-value)))


SITE_PROFILES = {
    "humid_tropical": {"temp": 28.0, "humidity": 78.0, "rain_scale": 2.4, "water": 42.0, "soil": 48.0},
    "river_plain": {"temp": 27.0, "humidity": 68.0, "rain_scale": 1.8, "water": 48.0, "soil": 40.0},
    "mountain_wet": {"temp": 20.0, "humidity": 76.0, "rain_scale": 2.5, "water": 30.0, "soil": 52.0},
    "semi_arid": {"temp": 33.0, "humidity": 38.0, "rain_scale": 0.55, "water": 22.0, "soil": 22.0},
    "urban_hot": {"temp": 34.0, "humidity": 50.0, "rain_scale": 0.9, "water": 28.0, "soil": 28.0},
    "forest_dry": {"temp": 31.0, "humidity": 35.0, "rain_scale": 0.65, "water": 24.0, "soil": 24.0},
}

EVENT_TYPES = [
    "flood",
    "landslide",
    "wildfire",
    "extreme_heat",
    "flood_landslide",
    "wildfire_extreme_heat",
]

HARD_NEGATIVE_SCENARIOS = [
    "rain_without_flood",
    "wet_stable_slope",
    "traffic_vibration",
    "cooking_smoke",
    "vehicle_exhaust",
    "hot_day_no_heatwave",
]

PROGRESS_MAP = {
    "pre_event": 0.20,
    "watch": 0.48,
    "warning": 0.75,
    "event": 1.00,
    "recovery": 0.45,
}


def profile_name(site_index, ood=False):
    standard_profiles = ["humid_tropical", "river_plain", "mountain_wet", "semi_arid", "urban_hot"]
    all_profiles = list(SITE_PROFILES.keys())
    if ood:
        return all_profiles[site_index % len(all_profiles)]
    return standard_profiles[site_index % len(standard_profiles)]


def start_state(profile):
    p = SITE_PROFILES[profile]
    return {
        "water_level_cm": clip(rng.normal(p["water"], 4), 4, 180),
        "rainfall_mm_h": clip(rng.exponential(p["rain_scale"]), 0, 100),
        "soil_moisture_pct": clip(rng.normal(p["soil"], 5), 5, 95),
        "temperature_c": clip(rng.normal(p["temp"], 3), 0, 55),
        "humidity_pct": clip(rng.normal(p["humidity"], 8), 5, 100),
        "tilt_x_deg": clip(rng.normal(0, 0.2), -10, 10),
        "tilt_y_deg": clip(rng.normal(0, 0.2), -10, 10),
        "acceleration_g": clip(rng.normal(1.0, 0.01), 0.7, 1.5),
        "smoke_raw": clip(rng.normal(100, 15), 0, 4000),
        "gas_raw": clip(rng.normal(200, 25), 0, 6000),
        "flame": 0,
        "flood_memory": 0.0,
        "landslide_memory": 0.0,
        "wildfire_memory": 0.0,
        "heat_memory": 0.0,
        "slide_direction_x": 0.0,
        "slide_direction_y": 0.0,
    }


MEMORY_KEYS = ["flood_memory", "landslide_memory", "wildfire_memory", "heat_memory"]
MEMORY_DECAY = 0.92        # per step once an event is over (~1 h half-life)
# How strongly each phase drives the event signals (memory target)
PHASE_INTENSITY = {"pre_event": 0.15, "watch": 0.35, "warning": 0.6, "event": 1.0}


def normal_step(previous, profile, timestamp):
    p = SITE_PROFILES[profile]
    minutes = timestamp.hour * 60 + timestamp.minute
    temp_cycle = 5.5 * math.sin(2 * math.pi * (minutes - 8 * 60) / (24 * 60))
    humidity_cycle = -9.0 * math.sin(2 * math.pi * (minutes - 8 * 60) / (24 * 60))

    rain_pulse = rng.exponential(p["rain_scale"])
    if rng.random() < 0.84:
        rain_pulse *= 0.10

    rainfall = clip(0.82 * previous["rainfall_mm_h"] + 0.18 * rain_pulse + rng.normal(0, 0.2), 0, 150)
    water_level = clip(
        previous["water_level_cm"]
        + 0.03 * rainfall
        - 0.018 * max(previous["water_level_cm"] - p["water"], 0)
        + rng.normal(0, 0.18),
        0,
        280,
    )
    # V4 fix: soil drains back towards the site's normal moisture
    # (the Colab version only ever got wetter and sat at 100 %).
    soil = clip(
        previous["soil_moisture_pct"]
        + 0.07 * rainfall * (1 - previous["soil_moisture_pct"] / 100)  # wet soil absorbs less
        - 0.02 * (previous["soil_moisture_pct"] - p["soil"])
        + rng.normal(0, 0.18),
        0,
        100,
    )
    temperature = clip(
        0.94 * previous["temperature_c"] + 0.06 * (p["temp"] + temp_cycle) + rng.normal(0, 0.18),
        -5,
        70,
    )
    humidity = clip(
        0.93 * previous["humidity_pct"]
        + 0.07 * (p["humidity"] + humidity_cycle)
        + 0.025 * rainfall
        + rng.normal(0, 0.35),
        2,
        100,
    )
    tilt_x = clip(0.995 * previous["tilt_x_deg"] + rng.normal(0, 0.02), -20, 20)
    tilt_y = clip(0.995 * previous["tilt_y_deg"] + rng.normal(0, 0.02), -20, 20)
    acceleration = clip(1.0 + rng.normal(0, 0.008), 0.7, 2.0)
    smoke = clip(0.93 * previous["smoke_raw"] + 0.07 * rng.normal(100, 15) + rng.normal(0, 2), 0, 6000)
    gas = clip(0.93 * previous["gas_raw"] + 0.07 * rng.normal(200, 25) + rng.normal(0, 3), 0, 9000)

    values = {
        "water_level_cm": water_level,
        "rainfall_mm_h": rainfall,
        "soil_moisture_pct": soil,
        "temperature_c": temperature,
        "humidity_pct": humidity,
        "tilt_x_deg": tilt_x,
        "tilt_y_deg": tilt_y,
        "acceleration_g": acceleration,
        "smoke_raw": smoke,
        "gas_raw": gas,
        "flame": 0,
        "slide_direction_x": previous.get("slide_direction_x", 0.0),
        "slide_direction_y": previous.get("slide_direction_y", 0.0),
    }
    # V4 fix: event memory carries over between steps and fades afterwards
    # (the Colab version reset it to 0 every step, so it never built up).
    for key in MEMORY_KEYS:
        values[key] = previous.get(key, 0.0) * MEMORY_DECAY
    return values


def pull(value, target, rate):
    """Move ``value`` a fraction ``rate`` of the way towards ``target``."""
    return value + rate * (target - value)


def apply_event_dynamics(values, event_type, phase_progress, profile, phase):
    """Event physics as bounded relaxation towards event targets.

    Memory (0..1) builds up through pre_event -> watch -> warning -> event and
    sets how far each signal is pulled from normal. Recovery applies no
    forcing; memory then decays in normal_step.
    """
    values = values.copy()
    if phase == "recovery":
        return values

    p = SITE_PROFILES[profile]
    target = PHASE_INTENSITY[phase]

    def build(key, speed=0.08):
        # memory was decayed in normal_step; undo that, then move towards the
        # phase's intensity so the signal peaks in the event phase itself
        return min(1.0, pull(values[key] / MEMORY_DECAY, target, speed))

    if "flood" in event_type:
        m = values["flood_memory"] = build("flood_memory")
        values["rainfall_mm_h"] = clip(pull(values["rainfall_mm_h"], 4 + 40 * m, 0.25) + rng.normal(0, 1.0), 0, 150)
        values["soil_moisture_pct"] = clip(values["soil_moisture_pct"] + 0.3 * m, 0, 100)
        values["water_level_cm"] = clip(
            pull(values["water_level_cm"], p["water"] + 160 * m, 0.10) + rng.normal(0, 0.3), 0, 280
        )

    if "landslide" in event_type:
        m = values["landslide_memory"] = build("landslide_memory")
        if values["slide_direction_x"] == 0.0:
            # A slope fails in one direction: keep it for the whole event
            angle = rng.uniform(0, 2 * math.pi)
            values["slide_direction_x"] = math.cos(angle)
            values["slide_direction_y"] = math.sin(angle)
        values["rainfall_mm_h"] = clip(pull(values["rainfall_mm_h"], 3 + 25 * m, 0.2), 0, 150)
        values["soil_moisture_pct"] = clip(
            pull(values["soil_moisture_pct"], p["soil"] + 30 + 15 * m, 0.15), 0, 100
        )
        creep = 0.005 + 0.06 * m ** 2
        values["tilt_x_deg"] = clip(values["tilt_x_deg"] + values["slide_direction_x"] * creep, -20, 20)
        values["tilt_y_deg"] = clip(values["tilt_y_deg"] + values["slide_direction_y"] * creep, -20, 20)
        values["acceleration_g"] = clip(1.0 + 0.12 * m ** 2 * abs(rng.normal(0, 1)) + rng.normal(0, 0.008), 0.7, 2.0)

    if "wildfire" in event_type:
        m = values["wildfire_memory"] = build("wildfire_memory")
        values["temperature_c"] = clip(pull(values["temperature_c"], p["temp"] + 22 * m, 0.15), 0, 75)
        values["humidity_pct"] = clip(pull(values["humidity_pct"], p["humidity"] * (1 - 0.6 * m), 0.15), 2, 100)
        values["smoke_raw"] = clip(
            pull(values["smoke_raw"], 100 + 2600 * m, 0.2) + rng.normal(0, 20 + 80 * m), 0, 6000
        )
        values["gas_raw"] = clip(
            pull(values["gas_raw"], 200 + 3200 * m, 0.2) + rng.normal(0, 25 + 100 * m), 0, 9000
        )
        values["flame"] = int(m > 0.65 and rng.random() < 0.5)

    if "extreme_heat" in event_type:
        # Heat waves build slowly
        m = values["heat_memory"] = build("heat_memory", speed=0.05)
        values["temperature_c"] = clip(pull(values["temperature_c"], p["temp"] + 6 + 9 * m, 0.08), 0, 70)
        values["humidity_pct"] = clip(pull(values["humidity_pct"], p["humidity"] - 20 * m, 0.08), 2, 100)

    return values


def apply_hard_negative(values, scenario, profile):
    """Look-alike situations that are NOT disasters.

    V4: each scenario is a contiguous episode (30 min - 3 h) that pulls the
    signals towards a plausible level, instead of a random single row that
    adds on top of the previous value (which made rain pile up).
    """
    values = values.copy()
    p = SITE_PROFILES[profile]

    if scenario == "rain_without_flood":
        values["rainfall_mm_h"] = clip(pull(values["rainfall_mm_h"], rng.uniform(10, 30), 0.3), 0, 150)
    elif scenario == "wet_stable_slope":
        values["rainfall_mm_h"] = clip(pull(values["rainfall_mm_h"], rng.uniform(5, 15), 0.3), 0, 150)
        values["soil_moisture_pct"] = clip(pull(values["soil_moisture_pct"], p["soil"] + 30, 0.1), 0, 100)
    elif scenario == "traffic_vibration":
        values["acceleration_g"] = clip(values["acceleration_g"] + rng.uniform(0.05, 0.25), 0.7, 2.0)
    elif scenario == "cooking_smoke":
        values["smoke_raw"] = clip(pull(values["smoke_raw"], rng.uniform(400, 900), 0.3), 0, 6000)
        values["gas_raw"] = clip(pull(values["gas_raw"], rng.uniform(300, 700), 0.3), 0, 9000)
    elif scenario == "vehicle_exhaust":
        values["smoke_raw"] = clip(pull(values["smoke_raw"], rng.uniform(200, 450), 0.3), 0, 6000)
        values["gas_raw"] = clip(pull(values["gas_raw"], rng.uniform(700, 1500), 0.3), 0, 9000)
    elif scenario == "hot_day_no_heatwave":
        values["temperature_c"] = clip(pull(values["temperature_c"], p["temp"] + rng.uniform(4, 8), 0.1), 0, 65)
        values["humidity_pct"] = clip(pull(values["humidity_pct"], p["humidity"] - 10, 0.1), 2, 100)

    return values


def schedule_states(total_steps, event_fraction):
    states = ["normal"] * total_steps
    target_event_steps = int(total_steps * event_fraction)
    used_steps = 0

    possible_starts = list(range(100, total_steps - 250))
    rng.shuffle(possible_starts)

    for start in possible_starts:
        if used_steps >= target_event_steps:
            break

        length = int(rng.integers(60, 180))
        end = min(start + length, total_steps)

        if any(states[index] != "normal" for index in range(start, end)):
            continue

        event_type = rng.choice(EVENT_TYPES)

        for index in range(start, end):
            p = (index - start) / max(length - 1, 1)
            if p < 0.20:
                phase = "pre_event"
            elif p < 0.48:
                phase = "watch"
            elif p < 0.78:
                phase = "warning"
            else:
                phase = "event"
            states[index] = f"{event_type}:{phase}"

        recovery_end = min(end + 30, total_steps)
        for index in range(end, recovery_end):
            if states[index] == "normal":
                states[index] = f"{event_type}:recovery"

        used_steps += length

    # Hard negatives: contiguous episodes covering ~15 % of the time
    target_negative_steps = int(0.15 * total_steps)
    negative_steps = 0
    attempts = 0
    while negative_steps < target_negative_steps and attempts < 20 * total_steps:
        attempts += 1
        length = int(rng.integers(6, 37))
        start = int(rng.integers(0, max(1, total_steps - length)))
        window = range(start, start + length)
        if any(states[index] != "normal" for index in window):
            continue
        scenario = rng.choice(HARD_NEGATIVE_SCENARIOS)
        for index in window:
            states[index] = "hard_negative:" + scenario
        negative_steps += length

    return states


def state_parts(state):
    if state == "normal":
        return "normal", "normal"
    return state.split(":", 1)


def simulate_site_truth(states, profile, site_start):
    """True environment + labels for one site, one row per 5-minute step."""
    rows = []
    previous = start_state(profile)

    for step, state in enumerate(states):
        timestamp = site_start + timedelta(minutes=step * C.SAMPLE_INTERVAL_MINUTES)
        kind, detail = state_parts(state)
        values = normal_step(previous, profile, timestamp)

        event_type = "normal"
        event_phase = "normal"
        if kind in EVENT_TYPES:
            event_type = kind
            event_phase = detail
            values = apply_event_dynamics(
                values, event_type, PROGRESS_MAP.get(event_phase, 0.0), profile, event_phase
            )
        elif kind == "hard_negative":
            values = apply_hard_negative(values, detail, profile)
            event_phase = "hard_negative"
        if "landslide" not in event_type:
            values["slide_direction_x"] = values["slide_direction_y"] = 0.0

        labels = {
            "flood_event": int("flood" in event_type and event_phase == "event"),
            "landslide_event": int("landslide" in event_type and event_phase == "event"),
            "wildfire_event": int("wildfire" in event_type and event_phase == "event"),
            "extreme_heat_event": int("extreme_heat" in event_type and event_phase == "event"),
        }
        rows.append({
            "timestamp": timestamp.isoformat(),
            **{k: v for k, v in values.items() if not k.endswith("_memory") and not k.startswith("slide_")},
            **labels,
            "is_disaster": int(max(labels.values()) == 1),
            "event_subtype": event_type,
            "event_phase": event_phase,
            "scenario_type": state,
        })
        previous = values.copy()

    truth = pd.DataFrame(rows)
    # V4 fix: "within N minutes" now means it. Label = 1 when the event phase
    # is happening now or starts within the horizon. The Colab labels were 1
    # from the start of the pre-event phase, i.e. hours ahead.
    for target, event_column, minutes in [
        ("flood_within_15m", "flood_event", 15),
        ("landslide_within_15m", "landslide_event", 15),
        ("wildfire_within_15m", "wildfire_event", 15),
        ("extreme_heat_within_60m", "extreme_heat_event", 60),
    ]:
        steps = minutes // C.SAMPLE_INTERVAL_MINUTES
        future = truth[event_column][::-1].rolling(steps + 1, min_periods=1).max()[::-1]
        truth[target] = future.astype(int)
    return truth


def generate_dataset(out_dir, file_name, role, site_prefix, sites, days,
                     event_fraction, start_time, failures, ood, ideal_sensors):
    frames = []
    total_steps = days * ROWS_PER_DAY
    label_columns = [
        "flood_event", "landslide_event", "wildfire_event", "extreme_heat_event",
        "flood_within_15m", "landslide_within_15m", "wildfire_within_15m",
        "extreme_heat_within_60m", "is_disaster", "event_subtype", "event_phase", "scenario_type",
    ]

    for site_index in range(sites):
        site_id = f"{site_prefix}_{site_index + 1:02d}"
        profile = profile_name(site_index + 1, ood)
        states = schedule_states(total_steps, event_fraction)
        site_start = start_time + timedelta(days=site_index * (days + 4))
        truth = simulate_site_truth(states, profile, site_start)

        for node_id in ["node1", "node2"]:
            if ideal_sensors:
                measured, fault_labels, delivered = SE.ideal_node(truth, node_id)
            else:
                scale = 3.0 if role == "locked_faults" else 1.0  # make the fault test set fault-heavy
                measured, fault_labels, delivered = SE.measure_node(truth, node_id, error_rng, failures, scale)
            tinyml = SE.tinyml_scores(measured, node_id, error_rng)

            frame = pd.concat(
                [truth[["timestamp"]], measured, tinyml, truth[label_columns]], axis=1
            )
            frame.insert(1, "node_id", node_id)
            frame.insert(2, "site_id", site_id)
            frame.insert(3, "sensor_status", np.where(fault_labels == "normal", "NORMAL", "FAULT"))
            frame["data_source"] = C.DATA_SOURCE
            frame["dataset_role"] = role
            frame["site_climate_profile"] = profile
            frame["sensor_fault_type"] = fault_labels
            frame["sensor_model"] = "ideal" if ideal_sensors else "realistic"
            frame["packet_delivered"] = delivered
            frames.append(frame)

    dataframe = pd.concat(frames, ignore_index=True)
    expected_rows = len(dataframe)
    dataframe = dataframe[dataframe.pop("packet_delivered")].reset_index(drop=True)
    packet_loss = 1 - len(dataframe) / expected_rows

    output_path = out_dir / file_name
    dataframe.to_csv(output_path, index=False)
    return dataframe, output_path, packet_loss


# (file key, role, site prefix, sites, days, event fraction, start, failures, ood)
# Site counts are sized so every held-out set holds roughly 30 episodes per
# hazard; the Colab sizes gave 2-7, too few for a meaningful PASS/FAIL.
DATASET_SPECS = [
    ("train", "train", "train_site", 10, 30, 0.34, datetime(2025, 1, 1), True, False),
    ("calibration", "calibration", "calibration_site", 10, 25, 0.16, datetime(2025, 12, 1), True, False),
    ("threshold_validation", "threshold_validation", "threshold_site", 12, 25, 0.13, datetime(2026, 3, 1), True, False),
    ("locked_normal", "locked_normal", "locked_normal_site", 20, 25, 0.08, datetime(2026, 6, 1), False, False),
    ("locked_faults", "locked_faults", "locked_fault_site", 20, 25, 0.08, datetime(2026, 9, 1), True, False),
    ("locked_ood", "locked_ood", "locked_ood_site", 16, 25, 0.10, datetime(2026, 11, 1), True, True),
]


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--out", type=Path, default=C.DATA_DIR, help="output folder")
    parser.add_argument("--quick", action="store_true",
                        help="small dataset (fewer sites/days) for a fast smoke test")
    parser.add_argument("--ideal-sensors", action="store_true",
                        help="perfect sensors (no noise, errors, failures or packet loss) for comparison")
    args = parser.parse_args()

    args.out.mkdir(parents=True, exist_ok=True)
    mode = "IDEAL sensors" if args.ideal_sensors else "REALISTIC sensors"
    print(f"Generating {C.DATA_SOURCE} datasets ({mode}) into {args.out}")

    manifest_rows = []
    for key, role, prefix, sites, days, fraction, start, failures, ood in DATASET_SPECS:
        if args.quick:
            sites = min(sites, 4 if key == "train" else 2)
            days = min(days, 10)
        started = time.time()
        dataframe, path, packet_loss = generate_dataset(
            args.out, C.DATASET_FILES[key], role, prefix, sites, days, fraction, start,
            failures, ood, args.ideal_sensors,
        )
        print(f"  {path.name}: {len(dataframe)} rows in {time.time() - started:.1f}s")

        manifest_rows.append({
            "file": path.name,
            "role": role,
            "rows": len(dataframe),
            "sites": dataframe["site_id"].nunique(),
            "sensor_model": "ideal" if args.ideal_sensors else "realistic",
            "packet_loss": round(packet_loss, 4),
            "fault_ratio": round((dataframe["sensor_status"] == "FAULT").mean(), 4),
            "flood_target_rate": round(dataframe["flood_within_15m"].mean(), 4),
            "landslide_target_rate": round(dataframe["landslide_within_15m"].mean(), 4),
            "wildfire_target_rate": round(dataframe["wildfire_within_15m"].mean(), 4),
            "extreme_heat_target_rate": round(dataframe["extreme_heat_within_60m"].mean(), 4),
            "data_source": C.DATA_SOURCE,
        })

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(args.out / C.MANIFEST_FILE, index=False)

    print(f"\n{C.DATA_SOURCE.upper()} DATASET MANIFEST")
    print(manifest_df.to_string(index=False))
    print("\nDATASET GENERATION COMPLETE")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Train -> calibrate -> pick thresholds -> locked tests. No forecasting.

Ported from the Colab "complete pipeline" with these changes:
* forecasting phase removed (all four forecasters scored worse than
  "repeat the last reading" in the Colab run);
* reads the files/roles produced by generate_dataset.py;
* models train on plain float32 arrays (HistGradientBoosting handles NaN
  itself), so live inference needs no pandas;
* the calibrator is stored as two floats, so applying it needs no sklearn;
* the calibrator no longer uses class_weight="balanced", which pulls
  probabilities towards 50/50 and defeats the point of calibrating;
* reports are CSV + JSON only (no Excel, no Colab downloads).

Usage:
    python3 train.py
    python3 train.py --data data --out output
"""

import argparse
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import config as C
from features import feature_matrix, prepare_dataframe
from metrics import (
    calculate_classification_metrics,
    print_classification_metrics,
    score_events,
    select_event_thresholds,
    site_groups,
)

REQUIRED_COLUMNS = (
    ["timestamp", "node_id", "site_id", "sensor_status"]
    + C.RAW_SENSOR_COLUMNS
    + C.TARGET_COLUMNS
    + ["dataset_role", "data_source", "event_subtype", "event_phase"]
)


def load_dataset(data_dir, key):
    path = data_dir / C.DATASET_FILES[key]
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run generate_dataset.py first.")

    header = pd.read_csv(path, nrows=0).columns
    missing = sorted(set(REQUIRED_COLUMNS) - set(header))
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")

    dataframe = pd.read_csv(path, usecols=REQUIRED_COLUMNS)

    roles = set(dataframe["dataset_role"].dropna().astype(str).unique())
    sources = set(dataframe["data_source"].dropna().astype(str).unique())
    if roles != {key}:
        raise ValueError(f"{path.name}: expected role={key}, found {roles}")
    if sources != {C.DATA_SOURCE}:
        raise ValueError(f"{path.name}: expected source={C.DATA_SOURCE}, found {sources}")

    print(f"{key}: {len(dataframe)} rows - PASS")
    return prepare_dataframe(dataframe)


def hazard_xy(dataframe, config):
    subset = dataframe[dataframe["node_id"] == config["node_id"]]
    return feature_matrix(subset, config["features"]), subset[config["target"]].to_numpy(dtype=int)


def hazard_groups(dataframe, config, hazard):
    return site_groups(dataframe[dataframe["node_id"] == config["node_id"]], hazard)


def count_episodes(dataframe, config):
    """Number of separate warning episodes (0 -> 1 transitions per site)."""
    subset = dataframe[dataframe["node_id"] == config["node_id"]]
    target = subset[config["target"]]
    previous = target.groupby(subset["site_id"]).shift(1).fillna(0)
    return int(((target == 1) & (previous == 0)).sum())


def fit_calibrator(raw_probabilities, y_true):
    inputs = np.clip(raw_probabilities, 1e-6, 1 - 1e-6).reshape(-1, 1)
    model = LogisticRegression(max_iter=1000, random_state=C.RANDOM_SEED)
    model.fit(inputs, y_true)
    return {"coef": float(model.coef_[0, 0]), "intercept": float(model.intercept_[0])}


def apply_calibrator(calibrator, raw_probabilities):
    inputs = np.clip(np.asarray(raw_probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return 1.0 / (1.0 + np.exp(-(calibrator["coef"] * inputs + calibrator["intercept"])))


def tag(metrics, **extra):
    metrics.update(extra)
    metrics["data_source"] = C.DATA_SOURCE
    return metrics


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", type=Path, default=C.DATA_DIR)
    parser.add_argument("--out", type=Path, default=C.OUTPUT_DIR)
    parser.add_argument("--max-iter", type=int, default=250, help="boosting iterations per model")
    args = parser.parse_args()

    model_dir = args.out / "models"
    report_dir = args.out / "reports"
    model_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    print("=" * 90)
    print("DISASTER PIPELINE: TRAIN -> CALIBRATION -> THRESHOLDS -> LOCKED TESTS")
    print("=" * 90)

    base_rows, calibration_rows, threshold_rows, locked_rows, acceptance_rows = [], [], [], [], []
    candidates = {}

    # Datasets are loaded one phase at a time and freed afterwards so the
    # whole pipeline fits in a Raspberry Pi's RAM.

    # ---- Phase 1: base classifiers --------------------------------------
    print("\nPHASE 1: BASE CLASSIFIERS")
    train_df = load_dataset(args.data, "train")
    models = {}
    for hazard, config in C.HAZARDS.items():
        X_train, y_train = hazard_xy(train_df, config)
        model = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=args.max_iter,
            max_leaf_nodes=15,
            l2_regularization=1.5,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=C.RANDOM_SEED,
        )
        model.fit(X_train, y_train)
        models[hazard] = model
        version = f"{hazard}_{C.MODEL_VERSION}"

        train_metrics = calculate_classification_metrics(y_train, model.predict_proba(X_train)[:, 1], 0.50)
        print_classification_metrics(f"{hazard.upper()} BASE TRAIN (iterations used: {model.n_iter_})", train_metrics)
        base_rows.append(tag(train_metrics, hazard=hazard, stage="BASE_TRAIN", dataset="train",
                             model_version=version, episodes=count_episodes(train_df, config)))
    del train_df

    # ---- Phase 2: calibration + ordered thresholds ----------------------
    print("\nPHASE 2: CALIBRATION + ORDERED THRESHOLDS")
    calibration_df = load_dataset(args.data, "calibration")
    threshold_df = load_dataset(args.data, "threshold_validation")
    for hazard, config in C.HAZARDS.items():
        model = models[hazard]
        version = f"{hazard}_{C.MODEL_VERSION}"
        X_cal, y_cal = hazard_xy(calibration_df, config)
        X_thr, y_thr = hazard_xy(threshold_df, config)

        empty = [name for name, y in [("calibration", y_cal), ("threshold_validation", y_thr)]
                 if len(np.unique(y)) < 2]
        if empty:
            reason = f"no positive (or no negative) rows in: {', '.join(empty)} - generate more data"
            print(f"{hazard.upper()} CANNOT CALIBRATE:", reason)
            acceptance_rows.append({"hazard": hazard, "candidate_status": "FAIL", "reason": reason})
            continue

        calibrator = fit_calibrator(model.predict_proba(X_cal)[:, 1], y_cal)
        cal_metrics = calculate_classification_metrics(
            y_cal, apply_calibrator(calibrator, model.predict_proba(X_cal)[:, 1]), 0.50
        )
        print_classification_metrics(f"{hazard.upper()} CALIBRATION", cal_metrics)
        calibration_rows.append(tag(cal_metrics, hazard=hazard, stage="CALIBRATION", dataset="calibration",
                                    model_version=version, episodes=count_episodes(calibration_df, config)))

        thr_probabilities = apply_calibrator(calibrator, model.predict_proba(X_thr)[:, 1])
        selection = select_event_thresholds(hazard_groups(threshold_df, config, hazard), thr_probabilities,
                                            C.ALERT_CONFIRM_READINGS)
        if selection["status"] == "FAIL":
            print(f"{hazard.upper()} THRESHOLD SELECTION FAILED:", selection["reason"])
            acceptance_rows.append({"hazard": hazard, "candidate_status": "FAIL", "reason": selection["reason"]})
            continue

        watch_threshold = selection["watch_threshold"]
        warning_threshold = selection["warning_threshold"]
        print(f"{hazard.upper()} thresholds: WATCH>={watch_threshold:.4f} WARNING>={warning_threshold:.4f} "
              f"(validation WARNING: {selection['warning_validation']})")
        if watch_threshold == warning_threshold:
            print(f"NOTE: {hazard} WATCH and WARNING thresholds are identical;"
                  " the two alert tiers will always fire together.")

        for stage, threshold in [("WATCH", watch_threshold), ("WARNING", warning_threshold)]:
            stage_metrics = calculate_classification_metrics(y_thr, thr_probabilities, threshold)
            print_classification_metrics(f"{hazard.upper()} THRESHOLD VALIDATION {stage}", stage_metrics)
            threshold_rows.append(tag(stage_metrics, hazard=hazard, stage=stage, dataset="threshold_validation",
                                      model_version=version, episodes=count_episodes(threshold_df, config)))

        candidates[hazard] = {
            "model": model,
            "calibrator": calibrator,
            "watch_threshold": watch_threshold,
            "warning_threshold": warning_threshold,
            "version": version,
        }
    del calibration_df, threshold_df

    # ---- Phase 3: locked tests ------------------------------------------
    print("\nPHASE 3: LOCKED TESTS")
    results = {hazard: {} for hazard in candidates}
    for test_name in ["locked_normal", "locked_faults", "locked_ood"]:
        test_df = load_dataset(args.data, test_name)
        for hazard, candidate in candidates.items():
            config = C.HAZARDS[hazard]
            X_test, y_test = hazard_xy(test_df, config)
            episodes = count_episodes(test_df, config)
            probabilities = apply_calibrator(candidate["calibrator"], candidate["model"].predict_proba(X_test)[:, 1])
            results[hazard][test_name] = {"episodes": episodes}
            groups = hazard_groups(test_df, config, hazard)
            for stage in ["WATCH", "WARNING"]:
                threshold = candidate[f"{stage.lower()}_threshold"]
                stage_metrics = calculate_classification_metrics(y_test, probabilities, threshold)
                events = score_events(groups, probabilities, threshold, C.ALERT_CONFIRM_READINGS)
                stage_metrics.update({f"event_{k}": v for k, v in events.items()})
                print(f"{hazard.upper()} {test_name} {stage}: {events}")
                print_classification_metrics(f"{hazard.upper()} {test_name.upper()} {stage} ({episodes} episodes)",
                                             stage_metrics)
                results[hazard][test_name][stage] = stage_metrics
                locked_rows.append(tag(dict(stage_metrics), hazard=hazard, stage=stage, dataset=test_name,
                                       model_version=candidate["version"], episodes=episodes))
        del test_df

    # ---- Acceptance gates + model export --------------------------------
    print("\nACCEPTANCE")
    registry = []
    for hazard, config in C.HAZARDS.items():
        model_path = model_dir / f"{hazard}_{C.MODEL_VERSION}.joblib"
        candidate = candidates.get(hazard)
        if candidate is None:
            # Never leave a model from an earlier run behind for a hazard that now fails
            model_path.unlink(missing_ok=True)
            continue

        res = results[hazard]
        normal_watch = res["locked_normal"]["WATCH"]
        normal_warning = res["locked_normal"]["WARNING"]
        detection = {name: res[name]["WARNING"]["event_detection_rate"] for name in res}
        fault_detection_drop = detection["locked_normal"] - detection["locked_faults"]
        ood_detection_drop = detection["locked_normal"] - detection["locked_ood"]
        min_episodes = min(res[name]["episodes"] for name in res)

        # Event-level gates decide PASS/FAIL. The per-reading Colab gates are
        # kept as row_* columns for reference.
        gates = {
            "ordered_thresholds_pass": candidate["watch_threshold"] <= candidate["warning_threshold"],
            "detection_rate_pass": detection["locked_normal"] >= C.MIN_DETECTION_RATE,
            "false_alarm_pass": normal_warning["event_false_alarms_per_site_day"]
            <= C.MAX_FALSE_ALARMS_PER_SITE_DAY,
            "alert_time_pass": normal_warning["event_alert_time_outside_events"]
            <= C.MAX_ALERT_TIME_OUTSIDE_EVENTS,
            "fault_robustness_pass": fault_detection_drop <= C.MAX_FAULT_DETECTION_DROP,
            "ood_robustness_pass": ood_detection_drop <= C.MAX_OOD_DETECTION_DROP,
            "brier_score_pass": normal_watch["brier_score"] <= C.MAX_BRIER_SCORE,
            "ece_pass": normal_watch["ece"] <= C.MAX_ECE,
        }
        gates = {name: bool(value) for name, value in gates.items()}
        row_gates = {
            "row_watch_recall_pass": bool(normal_watch["recall"] >= C.WATCH_RECALL_TARGET),
            "row_warning_precision_pass": bool(normal_warning["precision"] >= C.WARNING_PRECISION_TARGET),
            "row_pr_auc_pass": bool(normal_watch["pr_auc"]
                                    >= normal_watch["positive_rate_baseline"] + C.MIN_PR_AUC_MARGIN_ABOVE_BASELINE),
        }
        if min_episodes < C.MIN_TEST_EPISODES:
            status = "INSUFFICIENT_EVENTS"
        else:
            status = "PASS" if all(gates.values()) else "FAIL"

        print(f"\n{hazard.upper()} (fewest episodes in a locked test: {min_episodes})")
        for name, value in {**gates, **row_gates}.items():
            print(f"  {name}: {'PASS' if value else 'FAIL'}")
        print("  FINAL STATUS:", status)

        acceptance_rows.append({
            "hazard": hazard,
            "model_version": candidate["version"],
            "candidate_status": status,
            "min_test_episodes": min_episodes,
            "watch_threshold": candidate["watch_threshold"],
            "warning_threshold": candidate["warning_threshold"],
            "normal_detection_rate": detection["locked_normal"],
            "faults_detection_rate": detection["locked_faults"],
            "ood_detection_rate": detection["locked_ood"],
            "normal_warned_before_event": normal_warning["event_warned_before_event"],
            "normal_median_lead_min": normal_warning["event_median_lead_min"],
            "normal_false_alarms_per_site_day": normal_warning["event_false_alarms_per_site_day"],
            "watch_false_alarms_per_site_day": normal_watch["event_false_alarms_per_site_day"],
            "normal_alert_time_outside_events": normal_warning["event_alert_time_outside_events"],
            "watch_alert_time_outside_events": normal_watch["event_alert_time_outside_events"],
            "fault_detection_drop": fault_detection_drop,
            "ood_detection_drop": ood_detection_drop,
            "normal_row_watch_recall": normal_watch["recall"],
            "normal_row_warning_precision": normal_warning["precision"],
            "normal_row_warning_recall": normal_warning["recall"],
            "normal_pr_auc": normal_watch["pr_auc"],
            "normal_brier_score": normal_watch["brier_score"],
            "normal_ece": normal_watch["ece"],
            **gates,
            **row_gates,
        })

        joblib.dump({
            "model": candidate["model"],
            "calibrator": candidate["calibrator"],
            "hazard": hazard,
            "node_id": config["node_id"],
            "target": config["target"],
            "features": list(config["features"]),
            "watch_threshold": candidate["watch_threshold"],
            "warning_threshold": candidate["warning_threshold"],
            "status": status,
            "version": candidate["version"],
            "data_source": C.DATA_SOURCE,
            "sklearn_version": sklearn.__version__,
        }, model_path, compress=3)
        registry.append({
            "hazard": hazard,
            "file": model_path.name,
            "status": status,
            "watch_threshold": candidate["watch_threshold"],
            "warning_threshold": candidate["warning_threshold"],
        })

    # ---- Reports -------------------------------------------------------
    reports = {
        "base_training_metrics.csv": base_rows,
        "calibration_metrics.csv": calibration_rows,
        "threshold_validation_metrics.csv": threshold_rows,
        "locked_test_metrics.csv": locked_rows,
        "acceptance_gates.csv": acceptance_rows,
    }
    for file_name, rows in reports.items():
        pd.DataFrame(rows).to_csv(report_dir / file_name, index=False)

    with open(model_dir / "model_registry.json", "w") as handle:
        json.dump({
            "models": registry,
            "data_source": C.DATA_SOURCE,
            "sklearn_version": sklearn.__version__,
            "alert_confirm_readings": C.ALERT_CONFIRM_READINGS,
            "numpy_version": np.__version__,
            "python_version": platform.python_version(),
            "machine": platform.machine(),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }, handle, indent=2)

    print("\n" + "=" * 90)
    print("SUMMARY")
    print("=" * 90)
    order = {hazard: index for index, hazard in enumerate(C.HAZARDS)}
    for row in sorted(acceptance_rows, key=lambda r: order[r["hazard"]]):
        if "normal_detection_rate" not in row:
            print(f"{row['hazard'].upper()} | STATUS={row['candidate_status']} | {row.get('reason', '')}")
            continue
        print(
            f"{row['hazard'].upper()} | STATUS={row['candidate_status']} | episodes>={row['min_test_episodes']} | "
            f"WARNING>={row['warning_threshold']:.4f}: detected {row['normal_detection_rate']:.0%} "
            f"(faults {row['faults_detection_rate']:.0%}, OOD {row['ood_detection_rate']:.0%}), "
            f"before event {row['normal_warned_before_event']:.0%}, lead {row['normal_median_lead_min']:.0f} min, "
            f"false alarms {row['normal_false_alarms_per_site_day']:.3f}/site-day, "
            f"alert {row['normal_alert_time_outside_events']:.2%} of normal time | "
            f"WATCH>={row['watch_threshold']:.4f}: false alarms {row['watch_false_alarms_per_site_day']:.2f}/site-day, "
            f"alert {row['watch_alert_time_outside_events']:.2%}"
        )

    print(f"\nModels:  {model_dir}")
    print(f"Reports: {report_dir}")
    print(f"Total time: {time.time() - started:.1f}s")
    print(f"\nDATA SOURCE: {C.DATA_SOURCE.upper()} - REAL-WORLD PERFORMANCE NOT ESTABLISHED")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile event_metrics.py
"""Event-level scoring: what an operator actually experiences.

Per-row precision/recall count 5-minute readings. For an early-warning
system the questions are per disaster:

* detection rate  - share of disaster episodes with at least one alert
                    from the first precursor until the event ends
* warned before   - share of episodes alerted before the event phase began
* lead time       - minutes between the first alert and the start of the
                    actual event phase (positive = early warning)
* false alarms    - separate alert runs that touch no disaster episode
                    (or its recovery), per site per day

Usage:
    python3 event_metrics.py                                   # output/models on data/
    python3 event_metrics.py --models output_ideal/models --data data
"""

import argparse
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import config as C
from features import feature_matrix, prepare_dataframe
from metrics import score_events, site_groups
from train import apply_calibrator

def evaluate(models_dir, data_dir, dataset_key, level="WARNING", confirm=1):
    raw = pd.read_csv(Path(data_dir) / C.DATASET_FILES[dataset_key])
    dataframe = prepare_dataframe(raw)
    rows = []
    for hazard, config in C.HAZARDS.items():
        path = Path(models_dir) / f"{hazard}_{C.MODEL_VERSION}.joblib"
        if not path.exists():
            rows.append({"hazard": hazard, "dataset": dataset_key, "status": "NO_MODEL"})
            continue
        artifact = joblib.load(path)
        subset = dataframe[dataframe["node_id"] == config["node_id"]]
        probability = apply_calibrator(
            artifact["calibrator"],
            artifact["model"].predict_proba(feature_matrix(subset, artifact["features"]))[:, 1],
        )
        threshold = artifact["warning_threshold" if level == "WARNING" else "watch_threshold"]
        scores = score_events(site_groups(subset, hazard), probability, threshold, confirm)
        rows.append({"hazard": hazard, "dataset": dataset_key, "level": level, "threshold": threshold, **scores})
    return pd.DataFrame(rows)


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--models", type=Path, default=C.MODEL_DIR)
    parser.add_argument("--data", type=Path, default=C.DATA_DIR)
    parser.add_argument("--level", choices=["WATCH", "WARNING"], default="WARNING")
    parser.add_argument("--confirm", type=int, default=C.ALERT_CONFIRM_READINGS,
                        help="consecutive readings over threshold before alerting")
    parser.add_argument("--out", type=Path, default=None, help="optional CSV path")
    args = parser.parse_args()

    results = pd.concat(
        [evaluate(args.models, args.data, key, args.level, args.confirm) for key in ["locked_normal", "locked_faults", "locked_ood"]],
        ignore_index=True,
    )
    print(results.round(3).to_string(index=False))
    if args.out:
        results.to_csv(args.out, index=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_csv.py
"""Evaluate the trained models on one CSV (Colab "upload-one-CSV tester").

If the CSV has the target columns (flood_within_15m, ...), prints metrics
and PASS/FAIL gates per hazard. If not, writes predictions only.

Usage:
    python3 evaluate_csv.py path/to/readings.csv
    python3 evaluate_csv.py readings.csv --models output/models --out output/external_test_reports
"""

import argparse
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import config as C
from features import feature_matrix, prepare_dataframe
from metrics import (
    calculate_classification_metrics,
    confirm_alerts,
    print_classification_metrics,
    score_events,
    site_groups,
)
from train import apply_calibrator


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("csv", type=Path)
    parser.add_argument("--models", type=Path, default=C.MODEL_DIR)
    parser.add_argument("--out", type=Path, default=C.OUTPUT_DIR / "external_test_reports")
    args = parser.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)

    raw = pd.read_csv(args.csv)
    print(f"Test CSV: {args.csv} ({len(raw)} rows, {len(raw.columns)} columns)")

    missing = [c for c in ["timestamp", "node_id", "site_id"] if c not in raw.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")
    absent_sensors = [c for c in C.RAW_SENSOR_COLUMNS if c not in raw.columns]
    if absent_sensors:
        print("WARNING: sensor columns absent, treated as missing:", ", ".join(absent_sensors))

    labelled_targets = [h["target"] for h in C.HAZARDS.values() if h["target"] in raw.columns]
    dataframe = prepare_dataframe(raw)

    results = []
    for hazard, config in C.HAZARDS.items():
        model_path = args.models / f"{hazard}_{C.MODEL_VERSION}.joblib"
        print("\n" + "=" * 70)
        print("HAZARD:", hazard.upper())
        print("=" * 70)
        if not model_path.exists():
            print("No model file (threshold selection failed in training or train.py not run).")
            results.append({"hazard": hazard, "status": "NO_MODEL"})
            continue

        artifact = joblib.load(model_path)
        subset = dataframe[dataframe["node_id"] == config["node_id"]].copy()
        if subset.empty:
            print(f"No rows for {config['node_id']}.")
            results.append({"hazard": hazard, "status": "NO_ROWS"})
            continue

        probabilities = apply_calibrator(
            artifact["calibrator"],
            artifact["model"].predict_proba(feature_matrix(subset, artifact["features"]))[:, 1],
        )
        watch, warning = artifact["watch_threshold"], artifact["warning_threshold"]
        print(f"WATCH threshold: {watch:.4f}   WARNING threshold: {warning:.4f}   "
              f"confirm readings: {C.ALERT_CONFIRM_READINGS}")

        # Alert levels exactly as the live service raises them
        watch_alert = np.zeros(len(subset), dtype=bool)
        warning_alert = np.zeros(len(subset), dtype=bool)
        for _, index in subset.groupby("site_id").indices.items():
            watch_alert[index] = confirm_alerts(probabilities[index] >= watch, C.ALERT_CONFIRM_READINGS)
            warning_alert[index] = confirm_alerts(probabilities[index] >= warning, C.ALERT_CONFIRM_READINGS)

        if config["target"] not in labelled_targets:
            output = subset[["timestamp", "site_id", "node_id"]].copy()
            output[f"{hazard}_probability"] = probabilities
            output[f"{hazard}_level"] = np.where(warning_alert, "WARNING", np.where(watch_alert, "WATCH", "SAFE"))
            path = args.out / f"{hazard}_predictions.csv"
            output.to_csv(path, index=False)
            print(f"No labels in CSV - predictions saved to {path}")
            results.append({"hazard": hazard, "status": "PREDICTIONS_ONLY", "rows": len(output)})
            continue

        y_true = subset[config["target"]].to_numpy(dtype=int)
        watch_metrics = calculate_classification_metrics(y_true, probabilities, watch)
        warning_metrics = calculate_classification_metrics(y_true, probabilities, warning)
        print_classification_metrics("WATCH", watch_metrics)
        print_classification_metrics("WARNING", warning_metrics)

        events = {}
        if {"event_subtype", "event_phase"} <= set(subset.columns):
            # Simulator-style CSV: score whole disasters, as train.py does
            events = score_events(site_groups(subset, hazard), probabilities, warning, C.ALERT_CONFIRM_READINGS)
            print("EVENTS (WARNING):", {k: round(v, 4) for k, v in events.items()})
            gates = {
                "detection_rate_pass": events["detection_rate"] >= C.MIN_DETECTION_RATE,
                "false_alarm_pass": events["false_alarms_per_site_day"] <= C.MAX_FALSE_ALARMS_PER_SITE_DAY,
                "alert_time_pass": events["alert_time_outside_events"] <= C.MAX_ALERT_TIME_OUTSIDE_EVENTS,
                "brier_pass": watch_metrics["brier_score"] <= C.MAX_BRIER_SCORE,
                "ece_pass": watch_metrics["ece"] <= C.MAX_ECE,
            }
        else:
            # Only labels: fall back to per-reading gates
            gates = {
                "watch_recall_pass": watch_metrics["recall"] >= C.WATCH_RECALL_TARGET,
                "warning_precision_pass": warning_metrics["precision"] >= C.WARNING_PRECISION_TARGET,
                "pr_auc_pass": bool(watch_metrics["pr_auc"]
                                    >= watch_metrics["positive_rate_baseline"] + C.MIN_PR_AUC_MARGIN_ABOVE_BASELINE),
                "brier_pass": watch_metrics["brier_score"] <= C.MAX_BRIER_SCORE,
                "ece_pass": watch_metrics["ece"] <= C.MAX_ECE,
            }
        gates = {name: bool(value) for name, value in gates.items()}
        status = "PASS" if all(gates.values()) else "FAIL"
        print("GATES:", {k: "PASS" if v else "FAIL" for k, v in gates.items()}, "->", status)

        results.append({
            "hazard": hazard,
            "status": status,
            "watch_threshold": watch,
            "warning_threshold": warning,
            **{f"watch_{k}": v for k, v in watch_metrics.items()},
            **{f"warning_{k}": warning_metrics[k] for k in ["precision", "recall", "f1"]},
            **{f"event_{k}": v for k, v in events.items()},
            **gates,
        })

    results_df = pd.DataFrame(results)
    path = args.out / "external_test_results.csv"
    results_df.to_csv(path, index=False)
    print("\nSUMMARY")
    columns = ["hazard", "status", "event_detection_rate", "event_false_alarms_per_site_day",
               "watch_recall", "warning_precision"]
    print(results_df[[c for c in columns if c in results_df]].to_string(index=False))
    print("\nSaved:", path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile live_inference.py
"""Live hazard inference for the Raspberry Pi.

Keeps a rolling 3-day history per (site, node) in memory, builds the same
features used in training, and returns calibrated probabilities with a
SAFE / WATCH / WARNING level per hazard. A level is raised only after
``ALERT_CONFIRM_READINGS`` consecutive readings over its threshold.

Two ways to feed it readings:

  1. HTTP (for the Node.js dashboard or an MQTT bridge):
       python3 live_inference.py serve --port 5001
       curl -X POST localhost:5001/predict -d '{"node_id":"node1", ...}'
       curl localhost:5001/health

  2. JSON lines on stdin (one reading per line, one result per line):
       cat readings.jsonl | python3 live_inference.py stdin

A reading is a JSON object with ``node_id`` (node1/node2, NODE_01/NODE_02
also accepted), optional ``site_id`` and ``timestamp`` (ISO 8601, default:
now), and any of the sensor fields in config.RAW_SENSOR_COLUMNS. Missing
fields are treated as missing sensor values.
"""

import argparse
import json
import math
import sys
import threading
import warnings
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import joblib
import numpy as np

import config as C
from features import StreamingFeatureEngine

# Field names used by the existing Node.js simulator/dashboard. Only fields
# with the same physical meaning and unit are mapped. The dashboard's boolean
# ``rain`` cannot be turned into mm/h, so it is deliberately not mapped.
FIELD_ALIASES = {
    "water_level": "water_level_cm",
    "soil_moisture": "soil_moisture_pct",
    "temperature": "temperature_c",
    "humidity": "humidity_pct",
    "smoke": "smoke_raw",
    "gas": "gas_raw",
    "rainfall": "rainfall_mm_h",
    "acceleration": "acceleration_g",
}


def normalise_reading(reading):
    reading = dict(reading)
    for alias, column in FIELD_ALIASES.items():
        if alias in reading and column not in reading:
            reading[column] = reading.pop(alias)
    # A single "tilt" angle is treated as tilt_x with tilt_y = 0, so the
    # tilt magnitude matches; the x/y split itself is unknown.
    if "tilt" in reading and "tilt_x_deg" not in reading:
        reading["tilt_x_deg"] = reading.pop("tilt")
        reading.setdefault("tilt_y_deg", 0.0)
    return reading


class HazardPredictor:
    def __init__(self, model_dir):
        model_dir = Path(model_dir)
        registry_path = model_dir / "model_registry.json"
        if not registry_path.exists():
            raise FileNotFoundError(f"{registry_path} not found. Run train.py first.")

        with open(registry_path) as handle:
            self.registry = json.load(handle)

        self._check_versions()

        self.models = {}
        for entry in self.registry["models"]:
            self.models[entry["hazard"]] = joblib.load(model_dir / entry["file"])

        missing = sorted(set(C.HAZARDS) - set(self.models))
        if missing:
            print(f"[live_inference] no validated model for: {', '.join(missing)} "
                  "(threshold selection failed in train.py) - these hazards report UNAVAILABLE",
                  file=sys.stderr)

        self.engine = StreamingFeatureEngine()
        self.confirm = self.registry.get("alert_confirm_readings", C.ALERT_CONFIRM_READINGS)
        self.streaks = {}  # (site, node, hazard) -> [readings >= WATCH, readings >= WARNING]
        self.lock = threading.Lock()

    def _check_versions(self):
        import sklearn

        trained_with = self.registry.get("sklearn_version")
        if trained_with and trained_with != sklearn.__version__:
            warnings.warn(
                f"Models were trained with scikit-learn {trained_with} but this device has "
                f"{sklearn.__version__}. Pickled models may load wrongly or fail; retrain on "
                "this device (python3 train.py) or install the matching version.",
                RuntimeWarning,
            )

    def predict(self, reading):
        reading = normalise_reading(reading)
        site_id = str(reading.get("site_id", "default_site"))
        with self.lock:
            node_id, features = self.engine.update(reading)
            return self._predict(node_id, site_id, reading, features)

    def _predict(self, node_id, site_id, reading, features):
        hazards = {}
        for hazard, config in C.HAZARDS.items():
            if config["node_id"] != node_id:
                continue
            artifact = self.models.get(hazard)
            if artifact is None:
                hazards[hazard] = {"level": "UNAVAILABLE", "probability": None}
                continue

            vector = np.array(
                [[features.get(name, math.nan) for name in artifact["features"]]],
                dtype=np.float32,
            )
            raw = float(artifact["model"].predict_proba(vector)[0, 1])
            raw = min(max(raw, 1e-6), 1 - 1e-6)
            calibrator = artifact["calibrator"]
            probability = 1.0 / (1.0 + math.exp(-(calibrator["coef"] * raw + calibrator["intercept"])))

            # An alert needs `confirm` consecutive readings over the threshold,
            # exactly as in training/evaluation (filters single noisy readings).
            streak = self.streaks.setdefault((site_id, node_id, hazard), [0, 0])
            streak[0] = streak[0] + 1 if probability >= artifact["watch_threshold"] else 0
            streak[1] = streak[1] + 1 if probability >= artifact["warning_threshold"] else 0
            if streak[1] >= self.confirm:
                level = "WARNING"
            elif streak[0] >= self.confirm:
                level = "WATCH"
            else:
                level = "SAFE"

            hazards[hazard] = {
                "probability": round(probability, 4),
                "level": level,
                "model_status": artifact["status"],
                "model_version": artifact["version"],
            }

        if not hazards:
            return {"error": f"unknown node_id {reading.get('node_id')!r}; expected node1 or node2"}

        return {
            "node_id": node_id,
            "site_id": site_id,
            "timestamp": reading.get("timestamp"),
            "hazards": hazards,
            "data_source": C.DATA_SOURCE,
        }


def run_stdin(predictor):
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            result = predictor.predict(json.loads(line))
        except Exception as error:  # keep the stream alive on bad lines
            result = {"error": str(error)}
        print(json.dumps(result), flush=True)


def run_server(predictor, host, port):
    class Handler(BaseHTTPRequestHandler):
        def _send(self, status, payload):
            body = json.dumps(payload).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            if self.path == "/health":
                self._send(200, {
                    "status": "ok",
                    "models": {h: a["status"] for h, a in predictor.models.items()},
                    "unavailable": sorted(set(C.HAZARDS) - set(predictor.models)),
                })
            else:
                self._send(404, {"error": "not found"})

        def do_POST(self):
            if self.path != "/predict":
                self._send(404, {"error": "not found"})
                return
            try:
                length = int(self.headers.get("Content-Length", 0))
                payload = json.loads(self.rfile.read(length) or b"{}")
                if isinstance(payload, list):
                    self._send(200, [predictor.predict(item) for item in payload])
                else:
                    self._send(200, predictor.predict(payload))
            except Exception as error:
                self._send(400, {"error": str(error)})

        def log_message(self, fmt, *args):
            pass

    server = ThreadingHTTPServer((host, port), Handler)
    print(f"Disaster ML inference listening on http://{host}:{port} (POST /predict, GET /health)")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("mode", choices=["serve", "stdin"])
    parser.add_argument("--models", type=Path, default=C.MODEL_DIR)
    parser.add_argument("--host", default="127.0.0.1",
                        help="use 0.0.0.0 to accept LAN requests (no authentication!)")
    parser.add_argument("--port", type=int, default=5001)
    args = parser.parse_args()

    predictor = HazardPredictor(args.models)
    if args.mode == "stdin":
        run_stdin(predictor)
    else:
        run_server(predictor, args.host, args.port)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_feature_parity.py
"""Check that live (streaming) features match the training (pandas) features.

Training/serving skew is the most likely way for a model to work in the
notebook and misbehave on the device, so run this after any change to
features.py:

    python3 test_feature_parity.py                 # uses data/locked_faults
    python3 test_feature_parity.py some_file.csv --rows 3000
"""

import argparse
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import config as C
from features import StreamingFeatureEngine, prepare_dataframe

ALL_FEATURES = sorted({f for h in C.HAZARDS.values() for f in h["features"]})


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("csv", nargs="?", type=Path,
                        default=C.DATA_DIR / C.DATASET_FILES["locked_faults"])
    parser.add_argument("--rows", type=int, default=4000,
                        help="rows per node to replay (faults file has NaNs, spikes, stuck values)")
    args = parser.parse_args()

    raw = pd.read_csv(args.csv)
    first_site = raw["site_id"].iloc[0]
    raw = raw[raw["site_id"] == first_site]
    raw = raw.groupby("node_id", sort=False).head(args.rows).reset_index(drop=True)

    batch = prepare_dataframe(raw)
    engine = StreamingFeatureEngine()

    worst = 0.0
    mismatches = 0
    for _, row in batch.iterrows():
        reading = {c: row[c] for c in C.RAW_SENSOR_COLUMNS}
        reading.update(node_id=row["node_id"], site_id=row["site_id"], timestamp=row["timestamp"].isoformat())
        _, live = engine.update(reading)
        for name in ALL_FEATURES:
            a, b = float(row[name]), float(live[name])
            if math.isnan(a) and math.isnan(b):
                continue
            if math.isnan(a) != math.isnan(b) or not np.isclose(a, b, rtol=1e-6, atol=1e-6):
                mismatches += 1
                if mismatches <= 10:
                    print(f"MISMATCH {name} at {row['timestamp']} {row['node_id']}: batch={a} live={b}")
            else:
                worst = max(worst, abs(a - b))

    checked = len(batch) * len(ALL_FEATURES)
    print(f"Checked {checked} feature values over {len(batch)} rows; mismatches={mismatches}; "
          f"max abs diff={worst:.2e}")
    sys.exit(1 if mismatches else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_live_vs_batch.py
"""End-to-end check: the live service must raise the same alerts as evaluation.

Replays one site of a labelled CSV reading-by-reading through
live_inference.HazardPredictor and compares each SAFE/WATCH/WARNING level
with the batch pipeline (features -> model -> calibrator -> thresholds ->
consecutive-reading confirmation).

    python3 test_live_vs_batch.py                       # first locked_normal site
    python3 test_live_vs_batch.py some.csv --max-rows 2000
"""

import argparse
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import config as C
from features import feature_matrix, prepare_dataframe
from live_inference import HazardPredictor
from metrics import confirm_alerts
from train import apply_calibrator


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("csv", nargs="?", type=Path, default=C.DATA_DIR / C.DATASET_FILES["locked_normal"])
    parser.add_argument("--models", type=Path, default=C.MODEL_DIR)
    parser.add_argument("--max-rows", type=int, default=3000, help="readings to replay (both nodes)")
    args = parser.parse_args()

    raw = pd.read_csv(args.csv)
    raw = raw[raw["site_id"] == raw["site_id"].iloc[0]].sort_values(["timestamp", "node_id"]).head(args.max_rows)

    predictor = HazardPredictor(args.models)
    live = {}
    for _, row in raw.iterrows():
        reading = {key: (None if pd.isna(value) else value) for key, value in row.items()}
        for hazard, result in predictor.predict(reading)["hazards"].items():
            live.setdefault(hazard, []).append(result["level"])

    batch = prepare_dataframe(raw)
    failed = False
    for hazard, config in C.HAZARDS.items():
        path = args.models / f"{hazard}_{C.MODEL_VERSION}.joblib"
        if not path.exists():
            print(f"{hazard}: no model, skipped")
            continue
        artifact = joblib.load(path)
        subset = batch[batch["node_id"] == config["node_id"]]
        probability = apply_calibrator(
            artifact["calibrator"], artifact["model"].predict_proba(feature_matrix(subset, artifact["features"]))[:, 1]
        )
        warning = confirm_alerts(probability >= artifact["warning_threshold"], predictor.confirm)
        watch = confirm_alerts(probability >= artifact["watch_threshold"], predictor.confirm)
        expected = np.where(warning, "WARNING", np.where(watch, "WATCH", "SAFE"))
        agreement = float((expected == np.array(live[hazard])).mean())
        failed |= agreement < 1.0
        print(f"{hazard:13s} readings={len(expected)} live==batch {agreement:.2%} WARNING readings={int(warning.sum())}")

    sys.exit(1 if failed else 0)


if __name__ == "__main__":
    main()


## 2. Generate the dataset

Set `QUICK = True` for a 1-minute smoke test (its models are too weak to use).

In [ ]:
QUICK = False
!python generate_dataset.py {"--quick" if QUICK else ""}

## 3. Train, calibrate, choose thresholds, run locked tests

In [ ]:
!python train.py | tail -n 25

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
acceptance = pd.read_csv("output/reports/acceptance_gates.csv")
acceptance[[c for c in [
    "hazard", "candidate_status", "watch_threshold", "warning_threshold",
    "normal_detection_rate", "faults_detection_rate", "ood_detection_rate",
    "normal_warned_before_event", "normal_median_lead_min",
    "normal_false_alarms_per_site_day", "normal_alert_time_outside_events",
] if c in acceptance.columns]]

## 4. Per-disaster scores on every locked test

In [ ]:
!python event_metrics.py --out output/reports/event_metrics.csv

## 5. Check that the Pi gives identical results

In [ ]:
!python test_feature_parity.py
!python test_live_vs_batch.py

## 6. Download the Raspberry Pi bundle

`pi_models.zip` contains the trained models, the reports and
`requirements-pi.txt` pinned to this notebook's library versions. On the Pi,
unzip it inside the `ml_raspberry_pi` folder (it fills `output/`) and run
`pip install -r requirements-pi.txt`.

In [ ]:
import sklearn, numpy, shutil
# Only scikit-learn must match exactly for pickled models; numpy just needs
# the same major version line (numpy 2 pickles need numpy >= 2). Exact numpy
# pins often have no Raspberry Pi wheel.
numpy_floor = "2.0" if int(numpy.__version__.split(".")[0]) >= 2 else "1.24"
with open("requirements-pi.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\nnumpy>={numpy_floor}\npandas>=2.0\njoblib>=1.3\n")
shutil.copy("requirements-pi.txt", "output/requirements-pi.txt")
shutil.make_archive("/content/pi_models", "zip", ".", "output")
from google.colab import files
files.download("/content/pi_models.zip")

In [ ]:
# Optional: download the generated dataset as well (about 80-100 MB zipped)
DOWNLOAD_DATASET = False
if DOWNLOAD_DATASET:
    shutil.make_archive("/content/disaster_v4_dataset", "zip", ".", "data")
    files.download("/content/disaster_v4_dataset.zip")

## 7. Score your own labelled sensor data (the only real accuracy test)

Upload a CSV with `timestamp, node_id, site_id`, the sensor columns and the
label columns (`flood_within_15m`, ...). Without label columns you get
predictions only.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    !python evaluate_csv.py "{name}"